<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_8_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Protein design

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. Each group designs a different kind of protein against the same kind of question -
*does it work, and how do you know?* - and all three are judged with the same numbers:

* **GROUP 1: a nanobody.** Design single-domain antibodies (VHH) against PD-L1, the protein that tumours use
  to switch off T cells, with RFantibody. You design loops on a fixed immune scaffold.
* **GROUP 2: a mini-binder.** Design a small de novo protein against the *same* surface of PD-L1, in two
  different ways (diffusion and hallucination), and compare them. You design a whole new protein.
* **GROUP 3: a functional site.** Take a catalytic site out of an enzyme and build a new protein around it,
  with sequence design that sees the metal or the substrate. You design chemistry, not just binding.

Groups 1 and 2 attack the same target with two different technologies, so at the end you can compare them
directly: an antibody and a de novo mini-binder on the same epitope is one of the real choices of the field.

The notebook has two parts for each group:

1. **Running the designs**: each member runs a batch with his or her own parameters; the results of everybody
   end up in the group folder on Google Drive.
2. **The report**: the analysis of the results of the whole group, with your text. The code cells collect the
   results of all the members and make the figures; write your answers in the text cells, in your own words.

**You will present your work to the class** (about 10 minutes per group, followed by questions; the other
groups have not done your exercise). Design produces beautiful pictures and very few certainties: a good
presentation says how many designs were generated, how many survived, by which criteria, and what you would
do in the laboratory.

The cells that start with `%%group 1`, `%%group 2` or `%%group 3` run only for the group you choose below.

**Group:** 1 / 2 / 3

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

### Organising the work

One member creates a folder `Task8_group1` (or `Task8_group2`, `Task8_group3`) in Google Drive and shares it
with the rest of the group, with editing rights. Everybody adds a shortcut to it in his or her own `MyDrive`
(right click on the folder -> `Organise` -> `Add shortcut`). The designs run on the local disk of Colab and the
results (structures, metrics, figures) are copied to this folder; each run has its own sub-folder, so different
members never write the same files.

Design is a game of numbers: most designs fail the filters, so what matters is how many you generate and how
many survive. Divide the runs among the members (see the **Planning the work** table of your group), each in
his or her own Colab session, and start early. The free version of Colab disconnects after some time without
activity, and the installation takes a few minutes at the beginning of every session, so keep the tab open and
check it from time to time.

If Colab disconnects, run the setup cells again and then the cell that was interrupted: the results already
copied to the group folder are not lost.

### Planning the work

Rough times on a free Colab T4 (they depend on the machine you get, and the **installation must be repeated
every session**, which is the first thing to plan for). Divide the batches between the members: everybody
runs the same cells with different parameters.

| group | step | time |
|:--|:--|:--|
| all | installation (RFantibody, or RFdiffusion + AlphaFold2) | 10-15 min |
| all | the Boltz-2 environment, the first time | ~5 min |
| **1** | 10 nanobody designs (diffusion) | ~15-25 min |
| **1** | sequences (ProteinMPNN) + check with RF2, 2 per design | ~20 min |
| **1** | Boltz-2 on one design | ~3-5 min |
| **2** | 8 backbones with RFdiffusion | ~20-30 min |
| **2** | 8 trajectories of hallucination (AfDesign) | ~20-40 min |
| **2** | designability test, 4 sequences per backbone, for each arm | ~15 min per arm |
| **3** | 8 scaffolds around a site | ~20-30 min |
| **3** | designability test, 4 sequences per scaffold | ~10-15 min |
| **3** | LigandMPNN (8 sequences per design) | ~1-2 min |
| **3** | Boltz-2 with the ligand, one sequence | ~3-6 min |

Everything that costs time is **resumable**: a cell that has already produced its files says `already done`
and returns at once, so after a disconnection you re-run the setup and then the same cells, and only what is
missing is computed. Do not change the parameters of a batch half way through, or you will recompute it.

One member working alone needs roughly 1.5-2 hours per batch; a group of four that divides the parameters
collects four times more designs in the same time, which is exactly what this kind of work needs.

# Setup (run these cells at the start of every session)

In [ ]:
GROUP = 1          # 1 (nanobody), 2 (mini-binder), 3 (functional site)

In [ ]:
# a GPU is needed for every design tool used here: Runtime -> Change runtime type -> T4 GPU
import subprocess
out = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', out if out else 'NONE -- go to Runtime -> Change runtime type and choose a GPU')

In [ ]:
import os, shutil
WORK = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(WORK)

if os.path.isdir('/content'):                      # on Colab: results on Drive, calculations on the local disk
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PATH = f'/content/drive/MyDrive/Task8_group{GROUP}'
    os.makedirs(DRIVE_PATH, exist_ok=True)
else:                                              # on your own computer
    DRIVE_PATH = None
local_path = os.path.join(WORK, f'T08_group{GROUP}')
os.makedirs(local_path, exist_ok=True)

def sync_from_drive():
    """bring to this machine the results of the whole group (also those of the other members)"""
    if DRIVE_PATH:
        shutil.copytree(DRIVE_PATH, local_path, dirs_exist_ok=True)

sync_from_drive()
print('results of the group in', DRIVE_PATH or local_path)

In [ ]:
%pip install -q py3Dmol

The helper functions used everywhere below. They are short: read them, you will need to know what they
compute when you discuss your results.

Two warnings about **units**. ColabDesign reports `plddt` between 0 and 1 and the errors `pae`/`i_pae`
divided by 31; the literature uses pLDDT between 0 and 100 and pAE in Ångström. `read_designs()` converts
everything to the usual units, so the thresholds below are the ones you find in papers.

In [ ]:
import os, re, json, glob, time, pickle, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAE_SCALE = 31.0          # ColabDesign divides pae and i_pae by 31

# --- results of a design run -------------------------------------------------------

def read_designs(loc):
    """the table written by designability_test.py, converted to the usual units"""
    df = pd.read_csv(os.path.join(loc, 'mpnn_results.csv'), index_col=0)
    if df['plddt'].max() <= 1.5:                  # ColabDesign writes pLDDT between 0 and 1
        df['plddt'] = 100 * df['plddt']
    for c in ['pae', 'i_pae']:                    # these are already in Å (the script multiplies them)
        if c in df.columns and df[c].max() <= 1.0:
            df[c] = PAE_SCALE * df[c]
    df['pdb'] = [os.path.join(loc, 'all_pdb', f'design{int(d)}_n{int(n)}.pdb')
                 for d, n in zip(df['design'], df['n'])]
    return df

# the filters used in the literature for this kind of design
FILTERS = {'monomer': dict(plddt=('>', 80), rmsd=('<', 2.0)),
           'oligomer': dict(plddt=('>', 80), rmsd=('<', 2.0), i_pae=('<', 10)),
           'binder':   dict(plddt=('>', 80), rmsd=('<', 2.0), i_pae=('<', 10), i_ptm=('>', 0.5))}

def apply_filters(df, kind='binder', **overrides):
    """add a column pass_X for every filter and a column 'accepted' for designs that pass all of them"""
    thr = dict(FILTERS[kind]); thr.update({k: v for k, v in overrides.items()})
    df = df.copy()
    ok = np.ones(len(df), dtype=bool)
    for name, (op, value) in thr.items():
        if name not in df.columns:
            continue
        col = df[name] > value if op == '>' else df[name] < value
        df['pass_' + name] = col
        ok &= col.values
    df['accepted'] = ok
    return df, thr

def funnel(df, thr, title='', name=None):
    """how many designs survive each filter, applied one after the other (the design funnel)"""
    labels, counts = ['generated'], [len(df)]
    keep = np.ones(len(df), dtype=bool)
    for n, (op, v) in thr.items():
        if n not in df.columns:
            continue
        keep = keep & ((df[n] > v) if op == '>' else (df[n] < v)).values
        labels.append(f'{n} {op} {v}')
        counts.append(int(keep.sum()))
    fig, ax = plt.subplots(figsize=(1.3 * len(labels) + 1, 3.2))
    ax.bar(labels, counts, color=['0.7'] + ['#4878d0'] * (len(labels) - 1))
    for i, c in enumerate(counts):
        ax.text(i, c, str(c), ha='center', va='bottom')
    ax.set_ylabel('designs'); ax.set_title(title)
    ax.set_ylim(0, max(counts) * 1.2)
    plt.xticks(rotation=20, ha='right'); plt.tight_layout()
    print(f'success rate: {counts[-1]}/{counts[0]} = {100*counts[-1]/max(counts[0],1):.0f}%')
    if name:
        save_figure(fig, name)
    return fig

def oligomer_contigs(pdb):
    """the contigs of a symmetric oligomer for the designability test: one segment per chain.
    The test divides the residues it reads by `copies`, so it must be given every chain of the design."""
    ca = read_pdb(pdb)
    ca = ca[(ca.name == 'CA') & (~ca.hetero)]
    order = list(dict.fromkeys(ca.chain))
    return ':'.join(f'{int((ca.chain == c).sum())}-{int((ca.chain == c).sum())}' for c in order)

def chains_to_AB(pdb):
    """put the target (the longest chain) in chain A and the binder in chain B: the designability test
    assigns the chain letters by position, so a complex must really be named A and B."""
    a = read_pdb(pdb)
    target, binder = guess_chains(a)
    if (target, binder) == ('A', 'B') or binder is None:
        return False
    m = {target: 'A', binder: 'B'}
    a = a.copy()
    a['line'] = [l[:21] + m.get(l[21], l[21]) + l[22:] for l in a.line]
    write_pdb(a, pdb)
    return True

def binder_contigs(pdb):
    """the contigs of a two-chain design, as the designability test wants them: the first chain (the target)
    is kept, the second (the binder) is redesigned. Lengths are taken from the design itself."""
    ca = read_pdb(pdb)
    ca = ca[(ca.name == 'CA') & (~ca.hetero)]
    order = list(dict.fromkeys(ca.chain))                     # chains in the order of the file
    n = {c: int((ca.chain == c).sum()) for c in order}
    target, binder = order[0], order[1]
    return f'A1-{n[target]}:{n[binder]}-{n[binder]}'

def motif_contigs(pdb):
    """the contigs of a scaffolded motif, from the .trb that RFdiffusion writes next to the design:
    the motif residues are kept, the rest is redesigned (the lengths are those actually generated)."""
    trb = pickle.load(open(pdb.replace('.pdb', '.trb'), 'rb'))
    motif = sorted(r for _, r in trb['con_hal_pdb_idx'])
    ca = read_pdb(pdb)
    total = int(((ca.name == 'CA') & (~ca.hetero)).sum())
    parts, i = [], 1
    for r in motif:
        if r < i:
            continue
        end = r
        while end + 1 in motif:
            end += 1
        if r > i:
            parts.append(f'{r-i}-{r-i}')
        parts.append(f'A{r}-{end}')
        i = end + 1
    if i <= total:
        parts.append(f'{total-i+1}-{total-i+1}')
    return '/'.join(parts)

# --- reading and comparing structures (no external dependency) ----------------------

AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H',
       'ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W',
       'TYR':'Y','VAL':'V','MSE':'M'}

def read_pdb(filename):
    """atoms of a pdb file as a DataFrame (name, resname, chain, resi, x, y, z, element, hetero)"""
    rows = []
    for l in open(filename):
        if l.startswith(('ATOM', 'HETATM')):
            rows.append(dict(name=l[12:16].strip(), resname=l[17:20].strip(), chain=l[21],
                             resi=int(l[22:26]), x=float(l[30:38]), y=float(l[38:46]), z=float(l[46:54]),
                             element=l[76:78].strip() or l[12:16].strip()[0], hetero=l.startswith('HETATM'),
                             line=l.rstrip('\n')))
    return pd.DataFrame(rows)

def sequence_of(atoms, chain=None):
    sel = atoms[(atoms.name == 'CA') & (~atoms.hetero)]
    if chain:
        sel = sel[sel.chain == chain]
    return ''.join(AA3.get(r, 'X') for r in sel.resname)

def write_pdb(atoms, filename):
    with open(filename, 'w') as f:
        for l in atoms.line:
            f.write(l + '\n')
        f.write('END\n')

def kabsch(P, Q):
    """rotation and translation that superpose P on Q, and the rmsd after superposition"""
    Pc, Qc = P - P.mean(0), Q - Q.mean(0)
    V, S, Wt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(V @ Wt))
    R = V @ np.diag([1, 1, d]) @ Wt
    rot = Pc @ R
    return R, Q.mean(0) - P.mean(0) @ R, float(np.sqrt(((rot - Qc) ** 2).sum() / len(P)))

def ca_rmsd(pdb1, pdb2, chain1=None, chain2=None, resi1=None, resi2=None):
    """C-alpha rmsd after superposition; resi1/resi2 select the residues to use (e.g. a motif)"""
    def ca(f, ch, resi):
        a = read_pdb(f) if isinstance(f, str) else f
        a = a[(a.name == 'CA') & (~a.hetero)]
        if ch:
            a = a[a.chain == ch]
        if resi is not None:
            a = a[a.resi.isin(resi)]
        return a[['x', 'y', 'z']].values
    P, Q = ca(pdb1, chain1, resi1), ca(pdb2, chain2, resi2)
    n = min(len(P), len(Q))
    return kabsch(P[:n], Q[:n])[2]

def superpose_onto(mobile_pdb, ref_pdb, mobile_resi, ref_resi, out_pdb, mobile_chain=None, ref_chain=None,
                   atom_names=('N', 'CA', 'C', 'O')):
    """superpose a structure on another using the selected residues (paired in the order given),
    and write the moved structure. The whole backbone is used, not only the C-alpha: two residues give
    8 atoms, enough to fix the orientation, while two C-alpha alone would not."""
    m, r = read_pdb(mobile_pdb), read_pdb(ref_pdb)
    def ca(a, ch, resi):
        s = a[a.name.isin(atom_names) & (~a.hetero) & (a.resi.isin(list(resi)))]
        if ch:
            s = s[s.chain == ch]
        order = {n: i for i, n in enumerate(atom_names)}
        s = s.assign(_o=[order[n] for n in s.name]).sort_values(['resi', '_o'])
        return s[['x', 'y', 'z']].values
    P, Q = ca(m, mobile_chain, mobile_resi), ca(r, ref_chain, ref_resi)
    if len(P) != len(Q) or len(P) < 3:
        raise ValueError(
            f'cannot superpose {os.path.basename(mobile_pdb)} on {os.path.basename(ref_pdb)}: '
            f'{len(P)} and {len(Q)} atoms selected. Chains present: '
            f'{chain_sizes(m)} and {chain_sizes(r)} - pass mobile_chain/ref_chain to choose one.')
    R, t, rms = kabsch(P, Q)
    xyz = m[['x', 'y', 'z']].values @ R + t
    lines = []
    for l, (x, y, z) in zip(m.line, xyz):
        lines.append(f'{l[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{l[54:]}')
    with open(out_pdb, 'w') as f:
        f.write('\n'.join(lines) + '\nEND\n')
    print(f'{os.path.basename(out_pdb)}: motif rmsd {rms:.2f} A')
    return rms

def chain_sizes(pdb):
    """number of residues per chain"""
    a = read_pdb(pdb) if isinstance(pdb, str) else pdb
    a = a[(a.name == 'CA') & (~a.hetero)]
    return a.groupby('chain').size().sort_values(ascending=False).to_dict()

def guess_chains(pdb):
    """(target, binder): the longest chain is the target, the shortest the binder/nanobody"""
    s = chain_sizes(pdb)
    if len(s) < 2:
        return (list(s)[0], None) if s else (None, None)
    ch = list(s)
    return ch[0], ch[-1]

def contacts(pdb, chain_a, chain_b, cutoff=5.0):
    """residues of chain_a with at least one atom within cutoff of chain_b (the interface), ligands excluded"""
    a = read_pdb(pdb) if isinstance(pdb, str) else pdb
    a = a[~a.hetero]
    A, B = a[a.chain == chain_a], a[a.chain == chain_b]
    if len(A) == 0 or len(B) == 0:
        return []
    d = np.linalg.norm(A[['x','y','z']].values[:, None, :] - B[['x','y','z']].values[None, :, :], axis=-1)
    close = (d < cutoff).any(1)
    return sorted(int(r) for r in set(A.resi.values[close]))

# --- viewing -----------------------------------------------------------------------

def show(*pdbs, colors=('lightblue', 'salmon', 'palegreen', 'khaki'), sticks=None, spheres=True):
    """show one or more pdb files; sticks = list of (chain, resi) to show as sticks"""
    import py3Dmol
    view = py3Dmol.view(width=700, height=500)
    for i, p in enumerate(pdbs):
        view.addModel(open(p).read(), 'pdb')
        view.setStyle({'model': i}, {'cartoon': {'color': colors[i % len(colors)]}})
        if spheres:
            view.addStyle({'model': i, 'hetflag': True}, {'sphere': {'scale': 0.4}})
    for ch, resi in (sticks or []):
        view.addStyle({'chain': ch, 'resi': str(resi)}, {'stick': {'colorscheme': 'yellowCarbon'}})
    view.zoomTo()
    return view.show()

# --- an independent opinion: Boltz-2 ------------------------------------------------

def boltz_yaml(path, chains, ligand=None, msa='empty', affinity_of=None):
    """write a Boltz input: chains = {'A': sequence, ...}, ligand = ('B', 'ZN') or ('B', 'smiles:CCO')"""
    lines = ['version: 1', 'sequences:']
    for cid, seq in chains.items():
        lines += [f'  - protein:', f'      id: {cid}', f'      sequence: {seq}', f'      msa: {msa}']
    if ligand:
        cid, code = ligand
        key, val = ('smiles', code[7:]) if code.startswith('smiles:') else ('ccd', code)
        lines += [f'  - ligand:', f'      id: {cid}', f"      {key}: {val}"]
    if affinity_of:
        lines += ['properties:', '  - affinity:', f'      binder: {affinity_of}']
    open(path, 'w').write('\n'.join(lines) + '\n')
    return path

BOLTZ_ENV = 'boltz-env'          # Boltz-2 has its own environment (see install_boltz below)

def install_boltz(rerun=False):
    """Boltz-2 in an environment of its own: it pins numpy, scipy, dm-tree and torch to versions that have
    no wheels for the Python of Colab (dm-tree would have to be compiled). As for RFdiffusion, we install it
    next to the notebook with uv and use it as an external program. A few minutes and about 3 GB."""
    env = os.path.join(WORK, BOLTZ_ENV)
    boltz = os.path.join(env, 'bin', 'boltz')
    if os.path.isfile(boltz) and not rerun:
        return boltz
    if not shutil.which('uv'):
        subprocess.run('pip install -q uv', shell=True)
    print('building the Boltz-2 environment (a few minutes)...')
    subprocess.run(f'uv venv -q --python 3.12 "{env}"', shell=True)
    p = subprocess.run(f'uv pip install -q --python "{env}/bin/python" boltz',
                       shell=True, capture_output=True, text=True)
    if not os.path.isfile(boltz):
        print((p.stdout + p.stderr).strip()[-1500:])
        raise RuntimeError('could not install Boltz-2')
    print('Boltz-2 is ready')
    return boltz

def run_boltz(yaml_file, out_dir='boltz', samples=1, recycles=3, use_msa_server=False, rerun=False):
    """run Boltz-2 and return the confidence of the first model (and the path of the structure)"""
    name = os.path.basename(yaml_file).replace('.yaml', '')
    conf = os.path.join(out_dir, 'boltz_results_' + name, 'predictions', name, f'confidence_{name}_model_0.json')
    if rerun or not os.path.isfile(conf):
        cmd = (f'{install_boltz()} predict {yaml_file} --out_dir {out_dir} --output_format pdb '
               f'--diffusion_samples {samples} --recycling_steps {recycles} --override')
        if use_msa_server:
            cmd += ' --use_msa_server'
        t0 = time.time()
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if not os.path.isfile(conf):
            print(r.stdout[-2000:], r.stderr[-2000:])
            raise RuntimeError('boltz failed')
        print(f'{name}: {time.time()-t0:.0f} s')
    c = json.load(open(conf))
    c['pdb'] = conf.replace('confidence_', '').replace('.json', '.pdb')
    save_file_to_drive(conf); save_file_to_drive(c['pdb'])
    return c

# --- saving ------------------------------------------------------------------------

def save_file_to_drive(filename):
    """copy one file to the group folder on Google Drive (only this file: never a whole folder)"""
    if DRIVE_PATH and os.path.isfile(filename):
        relative = os.path.relpath(os.path.abspath(filename), local_path)
        if relative.startswith('..'):            # outside the group folder: not ours to copy
            return
        target = os.path.join(DRIVE_PATH, relative)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.copy2(filename, target)

def save_figure(fig, name):
    """save a figure for your presentation in the folder figures (also on Google Drive)"""
    os.makedirs(os.path.join(local_path, 'figures'), exist_ok=True)
    f = os.path.join(local_path, 'figures', name + '.png')
    fig.savefig(f, dpi=200, bbox_inches='tight')
    save_file_to_drive(f)
    print('saved figures/' + name + '.png')

def save_json(data, filename):
    os.makedirs(os.path.dirname(filename) or '.', exist_ok=True)
    def plain(o):                      # numpy types are not json
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)): return o.tolist()
        return str(o)
    json.dump(data, open(filename, 'w'), indent=1, default=plain)
    save_file_to_drive(filename)

def load_results(pattern, what='results'):
    """read into one table the json files of every member of the group (and say so if there are none)"""
    data = load_json(pattern)
    if not data:
        raise FileNotFoundError(
            f'no {what} found as {os.path.basename(pattern)}. Either nobody in the group has run that '
            f'part yet, or the group folder is not synchronised: run sync_from_drive() and look into '
            f'{os.path.dirname(pattern)}')
    return pd.concat([pd.DataFrame(v) for v in data.values()], ignore_index=True)

def load_json(pattern):
    """read all the json files matching a pattern, also those made by the other members of the group"""
    out = {}
    for f in sorted(glob.glob(pattern)):
        if ' (' in f:            # a duplicate made by Google Drive
            continue
        out[os.path.basename(f).replace('.json', '')] = json.load(open(f))
    return out

# --- the input structures ----------------------------------------------------------

def fetch_pdb(code, chains=None, keep_hetero=(), out=None, renumber=False, rename_chain=None):
    """download a structure from the PDB and keep what we need (chains, ligands, no water)"""
    code = code.lower()
    out = out or os.path.join(local_path, f'{code}.pdb')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    raw = os.path.join(os.path.dirname(out) or '.', code + '_raw.pdb')
    for attempt in range(2):
        if os.path.isfile(raw) and os.path.getsize(raw) > 1000:
            break
        subprocess.run(f'wget -q -O {raw} https://files.rcsb.org/download/{code.upper()}.pdb', shell=True)
    if not (os.path.isfile(raw) and os.path.getsize(raw) > 1000):
        raise RuntimeError(f'could not download {code.upper()} from the PDB (no network?)')
    seen, n = set(), 0
    with open(out, 'w') as f:
        for l in open(raw):
            if not l.startswith(('ATOM', 'HETATM')):
                continue
            if l[16] not in (' ', 'A'):                      # keep one conformation only
                continue
            ch, resn = l[21], l[17:20].strip()
            if chains and ch not in chains:
                continue
            if l.startswith('HETATM') and resn not in keep_hetero:
                continue
            if (ch, l[22:27]) not in seen:
                seen.add((ch, l[22:27])); n += 1
            if rename_chain:
                l = l[:21] + rename_chain.get(ch, ch) + l[22:]
            f.write(l)
        f.write('END\n')
    print(f'{out}: {n} residues')
    return out

In [ ]:
def run(cmd, log, rerun=False, done=None):
    """run a command, keep its output in a log file, do not repeat work already done"""
    if not rerun and done and all(os.path.exists(f) for f in done):
        print('already done:', ', '.join(os.path.basename(f) for f in done)); return
    t0 = time.time()
    os.makedirs(os.path.dirname(log) or '.', exist_ok=True)
    with open(log, 'w') as f:
        p = subprocess.run(cmd, shell=True, cwd=WORK, stdout=f, stderr=subprocess.STDOUT)
    if p.returncode or (done and not all(os.path.exists(f) for f in done)):
        print(open(log).read()[-3000:])
        raise RuntimeError(f'the command failed (exit code {p.returncode}), see ' + log)
    print(f'done in {time.time()-t0:.0f} s  (log: {os.path.basename(log)})')

In [ ]:
import linecache, warnings
from IPython import get_ipython
from IPython.display import display
from IPython.core.magic import register_cell_magic
warnings.filterwarnings('ignore')

@register_cell_magic
def group(line, cell):
    """a cell that starts with %%group N (or %%group 2 3) runs only for those groups"""
    ipython = get_ipython()
    groups = [int(x) for x in line.split()]
    if ipython.user_ns.get('GROUP') not in groups:
        print(f'This cell is for group {" and ".join(map(str, groups))}: skipped.')
        return
    code = ipython.transform_cell(cell)
    filename = f'<group {line} cell>'
    linecache.cache[filename] = (len(code), None, code.splitlines(True), filename)
    exec(compile(code, filename, 'exec'), ipython.user_ns)

Groups 2 and 3 install RFdiffusion, ProteinMPNN and AlphaFold2, as in the training (10-15 minutes: RFdiffusion
needs an environment with its own older Python, built next to the notebook); group 1 installs RFantibody
instead, in its own section below.

If the cell ends with an error, run it again: nothing already installed is downloaded twice, so a second run
only does what is missing.

In [ ]:
%%group 2 3
# RFdiffusion (backbones) + ColabDesign (ProteinMPNN and AlphaFold2). 10-15 minutes, once per session.
# The weights are about 5 GB and are downloaded on the virtual machine, not on your Drive.
#
# Two complications, both typical of research software:
#  * RFdiffusion needs the library dgl, which is not published for the Python of Colab, so we build next to
#    the notebook a small environment with an older Python (with uv, a fast installer) and run RFdiffusion
#    inside it as an external program. Everything else (ProteinMPNN, AlphaFold2, Boltz-2, the analysis) runs
#    in the notebook itself.
#  * the server that distributed the dgl packages (data.dgl.ai) has been unreachable since November 2026,
#    so we take the same build from the conda channel of the dgl team, which still works.
import os, re, sys, glob, time, shutil, subprocess

RFD_ENV = os.path.join(WORK, 'rfd-env')                    # the environment of RFdiffusion
RFD_PYTHON = os.path.join(RFD_ENV, 'bin', 'python')
RFD_REPO = os.path.join(WORK, 'RFdiffusion')
AF2_PARAMS = os.path.join(WORK, 'params')                  # where ColabDesign looks for AlphaFold2

def sh(cmd, tries=1, quiet=False):
    # run a shell command; on failure show the end of its output (os.system hides it)
    for t in range(tries):
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if p.returncode == 0:
            return True
        if t + 1 < tries:
            time.sleep(5)
    if not quiet:
        print('FAILED:', cmd)
        print((p.stdout + p.stderr).strip()[-1200:])
    return False

def imports(modules, python=None):
    # True if those modules can be imported by that interpreter
    return subprocess.run([python or sys.executable, '-c', 'import ' + ', '.join(modules)],
                          capture_output=True).returncode == 0

# --- code ------------------------------------------------------------------------------------------------
if not shutil.which('aria2c'):
    sh('apt-get -qq install aria2')
if not os.path.isfile(os.path.join(RFD_REPO, 'run_inference.py')):   # also repairs an interrupted clone
    shutil.rmtree(RFD_REPO, ignore_errors=True)
    sh(f'cd {WORK} && git clone -q https://github.com/sokrypton/RFdiffusion.git', tries=2)
    if not os.path.isfile(os.path.join(RFD_REPO, 'run_inference.py')):
        raise RuntimeError('could not clone RFdiffusion: check the network and run the cell again')
if not imports(['colabdesign']):
    sh('pip -q install git+https://github.com/sokrypton/ColabDesign.git', tries=2)
    sh('ln -sf /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign')
if not imports(['py3Dmol']):
    sh('pip install -q py3Dmol')

# --- weights (each file is downloaded only if missing: re-running this cell resumes, it does not restart) --
# RFdiffusion looks for its checkpoints in RFdiffusion/models: Base_ckpt for a monomer, an oligomer or a
# motif, Complex_base_ckpt as soon as hotspots are given (a binder). The noise schedules are computed by
# RFdiffusion itself the first time (a couple of minutes) and cached in RFdiffusion/schedules.
os.makedirs(f'{RFD_REPO}/models', exist_ok=True)
IPD = 'http://files.ipd.uw.edu/pub/RFdiffusion'
for ckpt, name in [('6f5902ac237024bdd0c176cb93063dc4', 'Base_ckpt.pt'),
                   ('e29311f6f1bf1af907f9ef9f44b8328b', 'Complex_base_ckpt.pt')]:
    if not os.path.isfile(f'{RFD_REPO}/models/{name}'):
        sh(f'aria2c -q -x 16 {IPD}/{ckpt}/{name} -d {RFD_REPO}/models', tries=2)
# AlphaFold2 (ColabDesign) looks for params/params_model_*.npz in the folder we work from
os.makedirs(AF2_PARAMS, exist_ok=True)
if not glob.glob(os.path.join(AF2_PARAMS, 'params_model_*.npz')):
    sh(f'cd {WORK} && aria2c -q -x 16 '
       f'https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar', tries=2)
    sh(f'tar -xf {WORK}/alphafold_params_2022-12-06.tar -C {AF2_PARAMS}')

# --- the environment of RFdiffusion ------------------------------------------------------------------------
DGL_CONDA = ('https://conda.anaconda.org/dglteam/label/th24_cu124/linux-64/'
             'dgl-2.4.0.th24.cu124-py{tag}_0.tar.bz2')          # dgl 2.4, torch 2.4, CUDA 12.4
DGL_WHEELS = 'https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html'   # the old way, kept in case it returns

def install_dgl(python_version):
    # the conda package contains lib/pythonX.Y/site-packages/dgl, exactly where the environment wants it
    url = DGL_CONDA.format(tag=python_version.replace('.', ''))
    print('  dgl from the conda channel (about 300 MB)')
    if sh(f'curl -sSL --max-time 900 -o /tmp/dgl.tar.bz2 {url}', tries=3, quiet=True):
        sh(f'tar -xjf /tmp/dgl.tar.bz2 -C "{RFD_ENV}" lib/python{python_version}/site-packages/dgl')
        sh('rm -f /tmp/dgl.tar.bz2')
        # libdgl.so looks for the CUDA libraries in <environment>/lib (that is its RPATH); they are the ones
        # that came with torch, inside site-packages/nvidia, so we point to them from there
        for so in (glob.glob(f'{RFD_ENV}/lib/python*/site-packages/nvidia/*/lib/lib*.so*')
                   + glob.glob(f'{RFD_ENV}/lib/python*/site-packages/torch/lib/lib*.so.*')):
            m = re.match(r'(lib[^-]+)-[0-9a-f]{6,}(\.so.*)$', os.path.basename(so))
            link = os.path.join(RFD_ENV, 'lib', m.group(1) + m.group(2) if m else os.path.basename(so))
            if not os.path.exists(link):
                os.symlink(so, link)
    if not imports(['dgl'], RFD_PYTHON):                        # fallback: the old wheel server
        print('  the conda package did not work, trying the old wheel server')
        sh(f'{UV} pip install -q --python "{RFD_PYTHON}" --no-deps dgl -f {DGL_WHEELS}', tries=2)
    return imports(['dgl'], RFD_PYTHON)

def check_rfd_env(verbose=True):
    p = subprocess.run([RFD_PYTHON, '-c',
                        'import torch, dgl, e3nn, hydra, se3_transformer; '
                        'print("torch", torch.__version__, "| dgl", dgl.__version__, '
                        '"| GPU", torch.cuda.is_available())'], capture_output=True, text=True)
    if verbose:
        print('  ' + (p.stdout.strip() or p.stderr.strip()[-700:]))
    return p.returncode == 0

def build_rfd_env(python_version):
    print(f'building the RFdiffusion environment with Python {python_version} (a few minutes)...')
    shutil.rmtree(RFD_ENV, ignore_errors=True)
    sh(f'{UV} venv -q --python {python_version} "{RFD_ENV}"', tries=2)
    add = f'{UV} pip install -q --python "{RFD_PYTHON}"'
    sh(f'{add} torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124', tries=2)
    sh(f'{add} "numpy<2" scipy sympy networkx pandas psutil requests tqdm pydantic packaging setuptools '
       f'omegaconf hydra-core icecream pynvml decorator pyrsistent', tries=2)
    install_dgl(python_version)
    sh(f'{add} --no-deps e3nn==0.5.5 opt_einsum opt_einsum_fx', tries=2)
    sh(f'{add} --no-deps "git+https://github.com/NVIDIA/dllogger#egg=dllogger"', tries=2)
    sh(f'{add} --no-deps {RFD_REPO}/env/SE3Transformer', tries=2)
    return check_rfd_env()

if not shutil.which('uv'):
    sh('pip install -q uv')
UV = shutil.which('uv') or 'uv'                 # the installer that builds the environments
if not (os.path.isfile(RFD_PYTHON) and check_rfd_env(verbose=False)):
    for version in ['3.12', '3.11']:
        if build_rfd_env(version):
            break

RUN_INFERENCE = os.path.join(RFD_REPO, 'run_inference.py')
print('RFdiffusion     :', len(glob.glob(f'{RFD_REPO}/models/*_ckpt.pt')), 'checkpoints')
print('AlphaFold2      :', len(glob.glob(os.path.join(AF2_PARAMS, 'params_model_*.npz'))),
      'weight files')
print('environment     :', end=' ')
if not check_rfd_env():
    raise RuntimeError('the environment of RFdiffusion is not working (see the messages above)')

In [ ]:
%%group 2 3
import colabdesign
COLABDESIGN = os.path.dirname(colabdesign.__file__)
DESIGNABILITY = os.path.join(COLABDESIGN, 'rf', 'designability_test.py')

def run_rfdiffusion(name, contigs, pdb=None, num_designs=4, steps=50, symmetry=None,
                    hotspot=None, partial_T=None, extra='', rerun=False):
    """generate backbones with RFdiffusion; returns the list of pdb files"""
    out = os.path.join(local_path, name, name)
    os.makedirs(os.path.dirname(out), exist_ok=True)
    opts = [f'inference.output_prefix={out}', f'inference.num_designs={num_designs}',
            f'diffuser.T={steps}',
            f"'contigmap.contigs=[{contigs}]'"]
    if pdb:       opts.append(f'inference.input_pdb={pdb}')
    if hotspot:   opts.append(f"'ppi.hotspot_res=[{hotspot}]'")
    if partial_T: opts.append(f'diffuser.partial_T={partial_T}')
    if symmetry:
        opts = ['--config-name symmetry', f'inference.symmetry={symmetry}',
                'potentials.guiding_potentials=[\\"type:olig_contacts,weight_intra:1,weight_inter:0.1\\"]',
                'potentials.olig_intra_all=True', 'potentials.olig_inter_all=True',
                'potentials.guide_scale=2', 'potentials.guide_decay=quadratic'] + opts
    files = [f'{out}_{i}.pdb' for i in range(num_designs)]
    run(f'DGLBACKEND=pytorch {RFD_PYTHON} {RUN_INFERENCE} ' + ' '.join(opts) + ' ' + extra,
        log=out + '_rfdiffusion.log', rerun=rerun, done=files)
    for f in files:
        save_file_to_drive(f); save_file_to_drive(f.replace('.pdb', '.trb'))
    return files

def run_designability(pdbs, name, contigs, copies=1, num_seqs=8, soluble=False, initial_guess=False,
                      multimer=False, recycles=3, temp=0.1, rerun=False):
    """ProteinMPNN + AlphaFold2 on each backbone; returns the table of metrics"""
    loc = os.path.join(local_path, name, 'designability')
    os.makedirs(loc, exist_ok=True)
    tables = []
    for pdb in sorted(pdbs):
        sub = os.path.join(loc, os.path.basename(pdb).replace('.pdb', ''))
        opts = (f'--pdb={pdb} --loc={sub} --contigs={contigs} --copies={copies} --num_seqs={num_seqs} '
                f'--num_recycles={recycles} --mpnn_sampling_temp={temp}')
        if soluble:       opts += ' --use_soluble'
        if initial_guess: opts += ' --initial_guess'
        if multimer:      opts += ' --use_multimer'
        run(f'python {DESIGNABILITY} {opts}', log=sub + '.log', rerun=rerun,
            done=[os.path.join(sub, 'mpnn_results.csv')])
        t = read_designs(sub)
        t.insert(0, 'backbone', os.path.basename(pdb).replace('.pdb', ''))
        tables.append(t)
    df = pd.concat(tables, ignore_index=True)
    for f in glob.glob(os.path.join(loc, '**', 'best*.pdb'), recursive=True):
        save_file_to_drive(f)      # only the best structures: copying hundreds of files to Drive is slow
    return df

# The assignment

Read the section of your group, then run its cells. The sections of the other groups are skipped
automatically, but read them too before the presentations.

# GROUP 1, Part 1: designing nanobodies

## The tool

RFantibody is RFdiffusion retrained on antibody structures. The difference with the general model matters:
an antibody does not invent a new fold, it keeps a **framework** (the immunoglobulin beta-sandwich, always the
same) and changes only the **CDR loops** that stick out of one end. A general backbone generator cannot do
this; RFantibody diffuses the loops while holding the framework fixed, and docks the whole thing on the
epitope you choose.

We design **nanobodies** (VHH, the single-domain antibodies of camelids): one chain instead of two, three CDR
loops instead of six, half the computer time. The framework comes with the program (`h-NbBCII10.pdb`, a
humanised camel nanobody).

The pipeline has the same three steps as in the training, with antibody-specific versions of each:

| step | program | what it does |
|:--|:--|:--|
| 1 | `rfdiffusion` | designs the CDR loops and the orientation of the nanobody on the epitope |
| 2 | `proteinmpnn` | writes the sequence of the loops (the framework keeps its own) |
| 3 | `rf2` | predicts the complex from the sequence and compares it with the design: keeps pAE < 10 and RMSD < 2 Å |

In [ ]:
%%group 1
# RFantibody: code, weights (about 1.5 GB) and its own Python 3.10 environment. 10-15 minutes, once per
# session. Two things are not as in the manual of RFantibody:
#  * it declares the library dgl with the address of a server (data.dgl.ai) that stopped answering in
#    November 2026, so `uv sync` fails; we take that dependency out and install the same build of dgl from
#    the conda channel of the dgl team, which still works;
#  * we call the programs through the environment directly (.venv/bin/rfdiffusion) instead of `uv run`,
#    which would try to repair the dependencies and undo the installation above.
import time

def sh(cmd, tries=1, quiet=False):
    # run a shell command; on failure show the end of its output (os.system hides it)
    for t in range(tries):
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if p.returncode == 0:
            return True
        if t + 1 < tries:
            time.sleep(5)
    if not quiet:
        print('FAILED:', cmd)
        print((p.stdout + p.stderr).strip()[-1200:])
    return False

def imports(modules, python):
    return subprocess.run([python, '-c', 'import ' + ', '.join(modules)], capture_output=True).returncode == 0

AB = os.path.join(WORK, 'RFantibody')
AB_PYTHON = os.path.join(AB, '.venv', 'bin', 'python')
FRAMEWORK = os.path.join(AB, 'scripts/examples/example_inputs/h-NbBCII10.pdb')   # nanobody (VHH)
WEIGHTS = ['RFdiffusion_Ab.pt', 'ProteinMPNN_v48_noise_0.2.pt', 'RF2_ab.pt']     # the names the code wants

# --- the code and the weights -----------------------------------------------------------------------------
# An interrupted clone leaves the folder behind but empty, and the download script of RFantibody skips
# everything if the weights folder merely exists. So we check for files that prove the work was done,
# and we download the weights one by one: running this cell again repairs whatever is missing.
if not os.path.isfile(os.path.join(AB, 'pyproject.toml')):
    shutil.rmtree(AB, ignore_errors=True)
    print('cloning RFantibody...')
    sh(f'cd {WORK} && git clone -q https://github.com/RosettaCommons/RFantibody.git', tries=2)
    if not os.path.isfile(os.path.join(AB, 'pyproject.toml')):
        raise RuntimeError('could not clone RFantibody: check the network and run the cell again')

os.makedirs(os.path.join(AB, 'weights'), exist_ok=True)
for w in WEIGHTS:
    target = os.path.join(AB, 'weights', w)
    if not (os.path.isfile(target) and os.path.getsize(target) > 10 ** 6):
        print('downloading', w, '...')
        sh(f'wget -q -O {target} https://files.ipd.uw.edu/pub/RFantibody/{w}', tries=2)

def dgl_package(python_tag, torch_version, cuda):
    """the dgl build of the conda channel that matches this environment, e.g.
    th23_cu118 / dgl-2.4.0.th23.cu118-py310_0 for torch 2.3.x with CUDA 11.8 and Python 3.10"""
    label = f"th{''.join(torch_version.split('+')[0].split('.')[:2])}_cu{cuda.replace('.', '')}"
    return (f'https://conda.anaconda.org/dglteam/label/{label}/linux-64/'
            f"dgl-2.4.0.{label.replace('_', '.')}-{python_tag}_0.tar.bz2")

def link_cuda_libraries(env):
    """libdgl.so looks for the CUDA libraries in <environment>/lib (its RPATH). They came with torch, either
    in site-packages/nvidia/*/lib or inside torch/lib with a hash in the name (libcudart-9335f6a2.so.12),
    which the loader does not recognise: we link them under the name it looks for."""
    n = 0
    for so in (glob.glob(f'{env}/lib/python*/site-packages/nvidia/*/lib/lib*.so*')
               + glob.glob(f'{env}/lib/python*/site-packages/torch/lib/lib*.so.*')):
        m = re.match(r'(lib[^-]+)-[0-9a-f]{6,}(\.so.*)$', os.path.basename(so))
        link = os.path.join(env, 'lib', m.group(1) + m.group(2) if m else os.path.basename(so))
        if not os.path.exists(link):
            os.symlink(so, link); n += 1
    return n

if not shutil.which('uv'):                                     # uv builds the environment
    sh('pip install -q uv')
UV = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')

if not (os.path.isfile(AB_PYTHON) and imports(['torch', 'dgl'], AB_PYTHON)):
    toml = os.path.join(AB, 'pyproject.toml')                  # remove the dgl dependency and its dead URL
    keep = [l for l in open(toml)
            if 'data.dgl.ai' not in l and l.strip().rstrip(',').strip('"') != 'dgl']
    open(toml, 'w').writelines(keep)
    sh(f'cd {AB} && {UV} sync', tries=2)

    # which environment did uv build? the dgl package must match the Python, the torch and the CUDA of torch
    info = subprocess.run([AB_PYTHON, '-c', 'import sys, torch; print(f"py{sys.version_info.major}'
                           '{sys.version_info.minor}", torch.__version__, torch.version.cuda)'],
                          capture_output=True, text=True)
    if info.returncode:
        print(info.stdout, info.stderr[-1000:])
        raise RuntimeError('uv sync did not build a working environment for RFantibody')
    PY_TAG, TORCH_VERSION, CUDA = info.stdout.split()
    PY_DIR = f'python{PY_TAG[2]}.{PY_TAG[3:]}'
    url = dgl_package(PY_TAG, TORCH_VERSION, CUDA)
    print(f'environment: {PY_TAG}, torch {TORCH_VERSION}, CUDA {CUDA}')
    print('installing dgl from the conda channel (about 300 MB):', url.split('/')[-1])
    shutil.rmtree(f'{AB}/.venv/lib/{PY_DIR}/site-packages/dgl', ignore_errors=True)   # a previous attempt
    for link in glob.glob(f'{AB}/.venv/lib/lib*.so*'):          # stale links to libraries that moved
        if not os.path.exists(os.path.realpath(link)):
            os.remove(link)
    if sh(f'curl -sSLf --max-time 900 -o /tmp/dgl_ab.tar.bz2 {url}', tries=3):
        sh(f'tar -xjf /tmp/dgl_ab.tar.bz2 -C {AB}/.venv lib/{PY_DIR}/site-packages/dgl')
        sh('rm -f /tmp/dgl_ab.tar.bz2')
        print('libdgl.so:', os.path.isfile(f'{AB}/.venv/lib/{PY_DIR}/site-packages/dgl/libdgl.so'))
    print('CUDA libraries linked for dgl:', link_cuda_libraries(os.path.join(AB, '.venv')))
    # dgl needs a few ordinary libraries that RFantibody does not install (psutil, networkx, tqdm...).
    # We add only the ones that are missing, so that nothing already in the environment is changed:
    # numpy in particular must stay below 2 for torch 2.3.
    needed = ['psutil', 'networkx', 'requests', 'tqdm', 'scipy', 'numpy', 'pandas', 'packaging', 'setuptools']
    missing = [m for m in needed if not imports([m], AB_PYTHON)]
    print('missing libraries for dgl:', missing or 'none')
    if missing:
        sh(f'{UV} pip install -q --python {AB_PYTHON} '
           + ' '.join('"numpy<2"' if m == 'numpy' else m for m in missing), tries=2)

# The three commands (rfdiffusion, proteinmpnn, rf2) are only launchers: each of them runs
# `python <script>` taking python from PATH. `uv run` would put the environment first in PATH (and would
# also undo our installation of dgl), so we do that part ourselves.
AB_RUN = f'cd {AB} && PATH={AB}/.venv/bin:$PATH VIRTUAL_ENV={AB}/.venv DGLBACKEND=pytorch'

if not os.path.isfile(os.path.join(WORK, 'USalign', 'USalign')):
    shutil.rmtree(os.path.join(WORK, 'USalign'), ignore_errors=True)
    sh(f'cd {WORK} && git clone -q https://github.com/pylelab/USalign.git && cd USalign && make -j2')

print('framework  :', os.path.isfile(FRAMEWORK))
have = [w for w in WEIGHTS if os.path.isfile(os.path.join(AB, 'weights', w))]
print('weights    :', f'{len(have)}/{len(WEIGHTS)} in RFantibody/weights',
      '' if len(have) == len(WEIGHTS) else '  MISSING: ' + ', '.join(set(WEIGHTS) - set(have)))
print('environment:', end=' ')
check = subprocess.run([AB_PYTHON, '-c', 'import torch, dgl; print("torch", torch.__version__, '
                        '"| dgl", dgl.__version__, "| GPU", torch.cuda.is_available())'],
                       capture_output=True, text=True)
print(check.stdout.strip() or '(the import did not finish)')

if check.returncode:
    # a failed import of dgl is almost always a missing or mismatched CUDA library: show everything
    print('\n--- the import ended with code', check.returncode,
          '(a negative code means the process was killed, usually a mismatched library) ---')
    print(check.stderr.strip()[-3000:] or '(no message at all)')
    libdgl = glob.glob(f'{AB}/.venv/lib/python*/site-packages/dgl/libdgl.so')
    if libdgl:
        not_found = subprocess.run(f'ldd {libdgl[0]} | grep "not found"', shell=True,
                                   capture_output=True, text=True).stdout.strip()
        print('\nlibraries that libdgl.so cannot find:\n' + (not_found or '  none'))
    else:
        print('\nlibdgl.so is not there: the conda package was not extracted')
    print('\nlinked in .venv/lib:', sorted(os.path.basename(f) for f in
                                           glob.glob(f'{AB}/.venv/lib/lib*.so*'))[:12])
    print('CUDA libraries that came with torch:',
          sorted({os.path.basename(os.path.dirname(os.path.dirname(f)))
                  for f in glob.glob(f'{AB}/.venv/lib/python*/site-packages/nvidia/*/lib')}) or 'none in site-packages/nvidia')
    raise RuntimeError('the environment of RFantibody is not working (see above). Run the cell again: '
                       'it repeats only what is missing.')

## The target: PD-L1

PD-L1 is the protein that tumours display to switch off the T cells that should kill them: it binds PD-1 on
the lymphocyte and delivers an "do not attack" signal. Antibodies that cover the PD-1 binding surface of PD-L1
(atezolizumab, durvalumab) are approved drugs, and so is a **nanobody**, envafolimab (KN035), the first
single-domain antibody used in the clinic. This is your target and these are your competitors.

We use the structure of the PD-1/PD-L1 complex, [4ZQK](https://www.rcsb.org/structure/4ZQK): chain A is PD-L1
(its IgV domain, residues 18-132), chain B is PD-1. We keep chain A only, renamed **T** (RFantibody wants the
target in chain T), and we take the surface used by PD-1 as the epitope to attack.

The **hotspots** are a few residues of that surface, used to tell the model where to dock the nanobody:
`T56, T58, T113, T122, T123` (Tyr56, Glu58, Arg113, Asp122, Tyr123). They are a suggestion, not a constraint:
part of your work is to check where the designs actually landed.

In [ ]:
%%group 1
target = fetch_pdb('4zqk', chains='A', out=os.path.join(local_path, 'pdl1_T.pdb'), rename_chain={'A': 'T'})
pdl1_seq = sequence_of(read_pdb(target))
print(pdl1_seq)
show(target, sticks=[('T', r) for r in [56, 58, 113, 122, 123]])    # yellow sticks: the hotspots

## Your runs

Each member designs a batch with **his or her own parameters** and the whole group compares them at the end.
Choose a different combination from the others, write it in the table of the report, and keep everything you
generate, failures included.

What you can change:

* `HOTSPOTS`: the five residues above, or a subset, or a different patch of PD-L1 (look at the structure and
  choose a surface; the PD-1 interface residues are 18-20, 23, 26, 54, 56, 58, 63, 66, 76, 113, 115, 117,
  119-126);
* `LOOPS`: the lengths of the three CDRs, above all H3, the long one that does most of the binding
  (`H3:5-13` lets the program choose between 5 and 13 residues; `H3:14-18` forces a long loop, as in many
  camelid nanobodies that bind into clefts);
* `NUM_DESIGNS`: how many backbones, and `NUM_SEQS`: how many sequences per backbone.

A batch of 10 designs takes roughly 20-30 minutes in total.

In [ ]:
%%group 1
MEMBER      = 'name'                        # your name: it labels your files in the group folder
HOTSPOTS    = 'T56,T58,T113,T122,T123'
LOOPS       = 'H1:7,H2:6,H3:5-13'
NUM_DESIGNS = 10                            # backbones (about 1-2 minutes each)
NUM_SEQS    = 2                             # sequences per backbone; every one of them costs an RF2
                                            # prediction (about a minute), so raise it only if you have time

run_dir = os.path.join(local_path, 'nanobody_' + MEMBER)
os.makedirs(run_dir, exist_ok=True)
print(run_dir)

In [ ]:
%%group 1
# step 1: the CDR loops and the dock on the epitope (about 1-2 minutes per design)
# RFantibody skips the designs it has already made, so re-running after a disconnection continues
run(f'{AB_RUN} .venv/bin/rfdiffusion -t {target} -f {FRAMEWORK} -o {run_dir}/diff/des -n {NUM_DESIGNS} '
    f'-l "{LOOPS}" -h "{HOTSPOTS}"',
    log=os.path.join(run_dir, 'rfdiffusion.log'),
    done=[os.path.join(run_dir, 'diff', f'des_{NUM_DESIGNS - 1}.pdb')])
backbones = sorted(glob.glob(os.path.join(run_dir, 'diff', 'des_*.pdb')))
print(len(backbones), 'backbones')
for f in backbones:
    save_file_to_drive(f)
show(backbones[0], target, colors=('salmon', 'lightblue')) if backbones else print(
    'no design was produced: look at', os.path.join(run_dir, 'rfdiffusion.log'))

In [ ]:
%%group 1
# step 2: the sequence of the loops. A nanobody has no light chain, so we ask only for the H loops
# (the default of the program is H1,H2,H3,L1,L2,L3, which would not exist here)
MPNN_LOOPS = ','.join(part.split(':')[0] for part in LOOPS.split(','))
run(f'{AB_RUN} .venv/bin/proteinmpnn -i {run_dir}/diff -o {run_dir}/mpnn -n {NUM_SEQS} -l "{MPNN_LOOPS}"',
    log=os.path.join(run_dir, 'mpnn.log'),
    done=[os.path.join(run_dir, 'mpnn', f'des_{len(backbones)-1}_dldesign_{NUM_SEQS-1}.pdb')])
print(len(glob.glob(os.path.join(run_dir, 'mpnn', '*.pdb'))), 'sequences')

# step 3: RF2 predicts the complex from the sequence and compares it with the design.
# This is the slow step (about 1 minute per sequence); it also skips what it has already done.
run(f'{AB_RUN} .venv/bin/rf2 -i {run_dir}/mpnn -o {run_dir}/rf2 -r 3',
    log=os.path.join(run_dir, 'rf2.log'))
predictions = sorted(glob.glob(os.path.join(run_dir, 'rf2', '*_best.pdb')))
print(len(predictions), 'predictions')
for f in predictions:
    save_file_to_drive(f)

RF2 writes one structure per sequence and, for each, the two numbers that decide its fate: the **pAE of the
interface** (how sure it is about the position of the nanobody on the target) and the **RMSD** between the
prediction and the design. The accepted thresholds are pAE < 10 and RMSD < 2 Å. The cell below collects them,
and adds what we compute ourselves: the length and the sequence of CDR H3, and which residues of PD-L1 each
design actually touches.

In [ ]:
%%group 1
def read_rfantibody(folder):
    """read what RF2 writes at the end of every predicted pdb: the metrics as `SCORE name: value`
    and the CDR loops as `REMARK PDBinfo-LABEL:  nn H3` (both come after the atoms)"""
    files = sorted(glob.glob(os.path.join(folder, '*_best.pdb')))      # the cycles are not written by default
    rows = []
    for f in files:
        name = os.path.basename(f)[:-len('_best.pdb')]
        d = {'design': name, 'backbone': name.split('_dldesign')[0], 'pdb': f}
        loops = {}
        for l in open(f):
            if l.startswith('SCORE'):
                m = re.match(r'SCORE\s+(\S+?):\s*(-?[\d.]+|nan)', l)
                if m and m.group(2) != 'nan':
                    d[m.group(1)] = float(m.group(2))
            elif 'PDBinfo-LABEL:' in l:
                m = re.search(r'PDBinfo-LABEL:\s*(\d+)\s+(\S+)', l)
                if m:
                    loops.setdefault(m.group(2), []).append(int(m.group(1)))
        for loop, residues in loops.items():
            d[loop + '_len'] = len(residues)
        d['loops'] = {k: sorted(v) for k, v in loops.items()}
        rows.append(d)
    df = pd.DataFrame(rows)
    # the names of the two filters of the RFantibody paper (pAE of the interface, and how far the
    # prediction is from the design once the target is superposed)
    for src, dst in [('interaction_pae', 'i_pae'), ('target_aligned_antibody_rmsd', 'rmsd'),
                     ('framework_aligned_cdr_rmsd', 'rmsd'), ('pred_lddt', 'plddt')]:
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]
    if 'plddt' in df.columns and df['plddt'].max() <= 1.5:
        df['plddt'] = 100 * df['plddt']
    return df

designs = read_rfantibody(os.path.join(run_dir, 'rf2'))
if len(designs) == 0:
    raise RuntimeError('no prediction in ' + os.path.join(run_dir, 'rf2'))
if 'i_pae' not in designs.columns:
    print('WARNING: no "SCORE" line found in the predictions; look inside one of the pdb files in',
          os.path.join(run_dir, 'rf2'))
print(len(designs), 'predictions from', designs.backbone.nunique(), 'backbones')

# what we add ourselves: the sequence of the CDR H3 and the epitope actually touched
h3_seq, epitope = [], []
for f, loops in zip(designs.pdb, designs.loops):
    atoms = read_pdb(f)
    target_chain, nb_chain = guess_chains(atoms)     # the long chain is PD-L1, the short one the nanobody
    ca = atoms[(atoms.name == 'CA') & (atoms.chain == nb_chain)]
    h3_seq.append(''.join(AA3.get(r, 'X') for r in ca[ca.resi.isin(loops.get('H3', []))].resname))
    epitope.append(contacts(atoms, target_chain, nb_chain))
designs['h3_seq'] = h3_seq
designs['epitope'] = epitope
designs['n_contacts'] = [len(e) for e in epitope]
designs['hotspots_hit'] = [len(set(e) & {56, 58, 113, 122, 123}) for e in epitope]
designs['member'] = MEMBER
designs['hotspots_asked'], designs['loops_asked'] = HOTSPOTS, LOOPS
save_json(designs.drop(columns=['pdb']).to_dict(orient='list'),
          os.path.join(local_path, 'results', f'nanobody_{MEMBER}.json'))
display(designs[[c for c in ['design', 'i_pae', 'rmsd', 'plddt', 'H3_len', 'n_contacts',
                             'hotspots_hit', 'h3_seq'] if c in designs.columns]].round(2))

## An independent opinion

As in the training, we ask a model that was not involved in the design. Boltz-2 receives the sequence of the
nanobody and the sequence of PD-L1 and predicts the complex from scratch: `iptm` says how confident it is
about the interface. Run it on your best two or three designs (about 3-5 minutes each); it is the slowest step,
so choose well.

In [ ]:
%%group 1
install_boltz()      # a few minutes the first time

In [ ]:
%%group 1
sel = designs.sort_values('i_pae').head(2) if 'i_pae' in designs else designs.head(2)
boltz_rows = []
for _, d in sel.iterrows():
    atoms = read_pdb(d.pdb)
    nb_seq = sequence_of(atoms, chain=guess_chains(atoms)[1])
    y = boltz_yaml(os.path.join(run_dir, d.design + '.yaml'),
                   chains={'H': nb_seq, 'T': pdl1_seq})
    c = run_boltz(y, out_dir=os.path.join(run_dir, 'boltz'))
    boltz_rows.append(dict(design=d.design, member=MEMBER, nanobody=nb_seq,
                           boltz_iptm=c['iptm'], boltz_plddt=100 * c['complex_plddt'],
                           rf2_i_pae=float(d.get('i_pae', float('nan'))), pdb=c['pdb']))
    print(d.design, 'Boltz-2 iptm', round(c['iptm'], 2))
boltz = pd.DataFrame(boltz_rows)
save_json(boltz.drop(columns=['pdb']).to_dict(orient='list'),
          os.path.join(local_path, 'results', f'boltz_{MEMBER}.json'))
display(boltz)

# GROUP 1, Part 2: the report

Write your answers in the text cells, in your own words, and put the figures where they are asked for. The
code cells collect the results of **all** the members from the group folder, so run `sync_from_drive()` first.

In [ ]:
%%group 1
sync_from_drive()
all_designs = load_results(os.path.join(local_path, 'results', 'nanobody_*.json'), 'designs')
print(len(all_designs), 'designs from', all_designs.member.nunique(), 'members')
display(all_designs.groupby(['member', 'loops_asked', 'hotspots_asked'])
        .size().rename('designs').reset_index())

## 1. What you did, and what came out

**Figure 1**: the procedure, and your best nanobody on PD-L1. Show the epitope and the CDR loops, and say in
the caption which parameters produced it.

In [ ]:
%%group 1
# the funnel of the whole group: generated -> predicted -> passes the two filters
thr = {'i_pae': ('<', 10), 'rmsd': ('<', 2.0)}
fig = funnel(all_designs, thr, title='nanobodies against PD-L1', name='funnel_group')

**Question 1.** What is the success rate of the whole group, and does it depend on the parameters? Compare the
members: longer H3 loops, different hotspots, more designs. Which choice helped?

*(your answer)*

## 2. Where do the nanobodies bind?

You asked for an epitope with the hotspots; the model is free to ignore you. The cell counts, over all the
designs of the group, how often each residue of PD-L1 is touched, and compares it with the surface used by
PD-1 (the natural partner) and by the nanobody KN035 (envafolimab), the drug.

In [ ]:
%%group 1
# the epitope used by PD-1 in the original complex, and by the nanobody KN035 (5JDS)
ref_pd1 = fetch_pdb('4zqk', chains='AB', out=os.path.join(local_path, '4zqk_AB.pdb'))
epi_pd1 = set(contacts(ref_pd1, 'A', 'B'))                       # PD-L1 (A) against PD-1 (B)
kn035 = fetch_pdb('5jds', out=os.path.join(local_path, '5jds.pdb'))
print('chains in 5JDS:', chain_sizes(kn035))                     # the longest chain is PD-L1

In [ ]:
%%group 1
PDL1_CHAIN, NB_CHAIN = guess_chains(kn035)   # the long chain is PD-L1, the short one the nanobody
epi_kn035 = set(contacts(kn035, PDL1_CHAIN, NB_CHAIN))
print(f'in 5JDS, PD-L1 is chain {PDL1_CHAIN} and the nanobody chain {NB_CHAIN}: '
      f'{len(epi_kn035)} residues of PD-L1 at the interface')
if not epi_kn035:
    print('WARNING: no contact found; check the chains above')

from collections import Counter
count = Counter(r for e in all_designs.epitope for r in e)
res = sorted(count)
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(res)), [count[r] for r in res], color='0.7')
for i, r in enumerate(res):
    if r in epi_pd1:   ax.bar(i, count[r], color='#4878d0')
    if r in epi_kn035: ax.plot(i, count[r] + 1, 'v', color='#ee854a')
ax.set_xticks(range(len(res))); ax.set_xticklabels(res, rotation=90, fontsize=6)
ax.set_xlabel('residue of PD-L1'); ax.set_ylabel('designs in contact')
ax.set_title('blue = used by PD-1, orange triangle = used by the nanobody KN035')
save_figure(fig, 'epitopes')

**Figure 2**: the figure above, plus a 3D view of one design superposed on KN035 (5JDS) or on PD-1.

**Question 2.** Did your designs land where you asked? Do they cover the surface that PD-1 uses, i.e. would
they block the interaction that matters? And how do they compare with the epitope of the real nanobody drug?

*(your answer)*

## 3. The loops

The CDR H3 is where the chemistry happens: in natural camelid nanobodies it is long (often 14-20 residues)
and frequently folds back onto the framework; in designs it is whatever you asked for.

In [ ]:
%%group 1
# the length of H3 and the size of the interface against the confidence of the prediction
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), sharey=True)
for m, d in all_designs.groupby('member'):
    if 'i_pae' in d and 'H3_len' in d:
        axes[0].scatter(d.H3_len + np.random.uniform(-.2, .2, len(d)), d.i_pae, label=m, alpha=0.7)
    if 'i_pae' in d:
        axes[1].scatter(d.n_contacts, d.i_pae, label=m, alpha=0.7)
for ax, xlabel in zip(axes, ['length of CDR H3', 'residues of PD-L1 touched']):
    ax.axhline(10, ls='--', c='k'); ax.set_xlabel(xlabel)
axes[0].set_ylabel('interface pAE (lower is better)'); axes[0].legend(fontsize=8)
plt.tight_layout(); save_figure(fig, 'loops_and_interface')

# the composition of the designed H3 loops, compared with what natural CDRs are made of
from collections import Counter
comp = Counter(''.join(all_designs.h3_seq.dropna().astype(str)))
print('amino acids used in the H3 loops:',
      ', '.join(f'{a}{100*n/max(sum(comp.values()),1):.0f}%' for a, n in comp.most_common(8)))

**Question 3.** Is there a relation between how much surface a design buries and how confident the prediction
is? Would you trust a design with very few contacts and an excellent pAE?

*(your answer)*

## 4. The second opinion

**Figure 3**: RF2 against Boltz-2 for the designs you tested with both.

In [ ]:
%%group 1
all_boltz = load_results(os.path.join(local_path, 'results', 'boltz_*.json'), 'Boltz-2 predictions')
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(all_boltz.rf2_i_pae, all_boltz.boltz_iptm, c='#4878d0')
ax.axvline(10, ls='--', c='k'); ax.axhline(0.5, ls='--', c='k')
ax.set_xlabel('RF2 interface pAE (lower is better)'); ax.set_ylabel('Boltz-2 iptm (higher is better)')
save_figure(fig, 'rf2_vs_boltz')
display(all_boltz.drop(columns=['pdb'], errors='ignore').round(3))

**Question 4.** Do the two models agree? If a design passes the RFantibody filters but Boltz-2 is not
confident about it, what would you conclude, and what would you do next?

*(your answer)*

## 5. Are they new, and would you make them?

Use [search.foldseek.com](https://search.foldseek.com) on your best nanobody (the whole structure, then only
the chain H). A nanobody is *supposed* to look like a nanobody, so the question is not whether the fold is new,
but whether the **sequence** is: compare it with the closest natural nanobody that Foldseek finds.

**Question 5.** Suppose you can order three designs for the laboratory. Which three do you choose, with which
criteria, and what experiment would tell you whether they work? Mention at least one property that none of the
numbers in this notebook measures (expression, solubility, aggregation, immunogenicity, affinity).

*(your answer)*

## 6. Going further (optional)

* Design with an **antibody** framework (`hu-4D5-8_Fv.pdb`, six CDRs) instead of a nanobody, with the same
  epitope: the authors of RFantibody note that nanobodies tend to dock "side-on" because that is how natural
  nanobodies bind. Do you see the difference?
* Take your best design and generate variations with `partial diffusion` on the loops only.
* Compare the amino acid composition of your CDRs with that of natural antibodies: do the designs use the
  tyrosines and serines that real CDRs are made of?

# GROUP 2, Part 1: two ways of making a binder

A **mini-binder** is a small protein (50-80 residues), usually a bundle of helices, designed to stick to one
patch of a target protein. It is not an antibody: it has no loops, no framework and no immune system behind
it. It is cheap to produce, very stable, and it is what the field uses when it wants a binder quickly.

There are two ideas for making one, and you will run both on the same target.

**Diffusion (RFdiffusion).** Generate a backbone next to the chosen patch, write a sequence for it with
ProteinMPNN, check with AlphaFold2. The three steps are separate and each can be replaced.

**Hallucination (AfDesign).** Do not generate anything: take a random sequence, ask AlphaFold2 to predict the
complex, define a *loss* that is small when the binder is folded and in contact with the target, and change
the sequence by gradient descent until AlphaFold2 itself predicts what you want. The structure prediction
network is used as a function to optimise. **BindCraft**, which holds the best published experimental success
rates for binders, is the industrial version of this idea.

The two give different proteins, fail in different ways and are judged by the same numbers. That comparison
is the point of this exercise.

| | diffusion | hallucination |
|:--|:--|:--|
| what is optimised | nothing: the backbone is sampled from the model | the sequence, against the prediction |
| the sequence comes from | ProteinMPNN | the optimisation itself (ProteinMPNN afterwards, to clean it up) |
| judge | AlphaFold2, which did not take part | AlphaFold2, which did take part |
| known weakness | many backbones are not designable | sequences that satisfy AlphaFold2 without being real proteins ("adversarial") |

## The target: PD-L1

PD-L1 is what tumours display to switch off the T cells that should kill them: it binds PD-1 on the lymphocyte
and delivers a "do not attack" signal. Antibodies that cover that surface (atezolizumab, durvalumab) are
approved drugs. Your group attacks the same surface with a de novo protein, and **group 1 attacks it with a
nanobody**: at the end you will compare the two modalities on the same epitope.

We use [4ZQK](https://www.rcsb.org/structure/4ZQK) (PD-1/PD-L1): chain A is PD-L1 (IgV domain, residues
18-132), renamed **T**. Hotspots on the PD-1 facing surface: `T56, T58, T113, T122, T123`.

There is also a published de novo binder against this target, [8ZNL](https://www.rcsb.org/structure/8ZNL)
(58 residues, picomolar affinity, 1.77 Å): your reference point.

In [ ]:
%%group 2
target = fetch_pdb('4zqk', chains='A', out=os.path.join(local_path, 'pdl1_T.pdb'), rename_chain={'A': 'T'})
pdl1_seq = sequence_of(read_pdb(target))
print(pdl1_seq)
show(target, sticks=[('T', r) for r in [56, 58, 113, 122, 123]])

In [ ]:
%%group 2
MEMBER      = 'name'                        # your name: it labels your files in the group folder
HOTSPOTS    = 'T56,T58,T113,T122,T123'
BINDER_LEN  = 65                            # 55-80; take a different length from the other members
assert BINDER_LEN < 110, 'keep the binder shorter than PD-L1 (115 residues): the chains are told apart by length'
NUM_DESIGNS = 8                             # backbones (arm A) / trajectories (arm B)
NUM_SEQS    = 4                             # sequences per backbone: AlphaFold2 predicts the complex for
                                            # each one, in both arms, so this doubles the cost twice over

run_dir = os.path.join(local_path, f'binder_{MEMBER}_{BINDER_LEN}')
os.makedirs(run_dir, exist_ok=True)
print(run_dir)

## Arm A: diffusion

The contigs say: keep residues 18-132 of chain T, then a chain break, then a new chain of `BINDER_LEN`
residues. The hotspots bias the docking. About 2-4 minutes per backbone, then about 1 minute per backbone for
the sequences and the check.

Note the two extra options in the designability test: `initial_guess` lets AlphaFold2 start from the designed
complex instead of from nothing (this is the standard protocol for binders, and it is much more severe:
without it everything looks good), and `use_multimer` uses the AlphaFold-multimer weights, trained on
complexes.

In [ ]:
%%group 2
backbones = run_rfdiffusion(f'diffusion_{MEMBER}', contigs=f'T18-132/0 {BINDER_LEN}-{BINDER_LEN}',
                            pdb=target, hotspot=HOTSPOTS, num_designs=NUM_DESIGNS, steps=50)
show(backbones[0], colors=('salmon',))

In [ ]:
%%group 2
# the contigs of the designability test: chain A (PD-L1) kept, chain B (the binder) redesigned
for f in backbones:
    chains_to_AB(f)
print('chains:', chain_sizes(backbones[0]), '| contigs:', binder_contigs(backbones[0]))
diff = run_designability(backbones, f'diffusion_{MEMBER}', contigs=binder_contigs(backbones[0]),
                         num_seqs=NUM_SEQS, initial_guess=True, multimer=True)
diff['method'] = 'diffusion'
display(diff[['backbone', 'n', 'mpnn', 'plddt', 'i_ptm', 'i_pae', 'rmsd']].round(2))

## Arm B: hallucination

AfDesign optimises the sequence of the binder through AlphaFold2 in three stages (continuous, then
intermediate, then discrete amino acids). Each trajectory gives one binder; they are independent, so run
several. About 2-5 minutes per trajectory on a T4.

Then we do something important for the comparison: we pass the hallucinated complexes through **the same**
ProteinMPNN + AlphaFold2 test as arm A. Judging a hallucinated design with the network that designed it would
be meaningless.

In [ ]:
%%group 2
from colabdesign import mk_afdesign_model, clear_mem

# the model is built once (that is the slow part, a minute of compilation) and restarted for every
# trajectory: `restart` draws a new random sequence, which is where a trajectory begins
clear_mem()
af = mk_afdesign_model(protocol='binder')
af.prep_inputs(pdb_filename=target, chain='T', binder_len=BINDER_LEN, hotspot=HOTSPOTS)
print('target', af._target_len, 'residues, binder', af._binder_len, '| hotspots at positions',
      af.opt['hotspot'])

halluc, halluc_log = [], []
for i in range(NUM_DESIGNS):
    out = os.path.join(run_dir, f'halluc_{i}.pdb')
    if os.path.isfile(out):                       # already done in an earlier session
        halluc.append(out); continue
    af.restart(seed=i)
    af.design_3stage(50, 50, 10)
    af.save_pdb(out)
    log = {k: float(v) for k, v in getattr(af, 'aux', {}).get('log', {}).items() if np.isscalar(v)}
    seq = (af.get_seqs() if hasattr(af, 'get_seqs') else af.get_seq())[0]
    print(i, seq, {k: round(v, 2) for k, v in log.items()})
    halluc.append(out); halluc_log.append(dict(trajectory=i, seq=seq, **log))
    save_file_to_drive(out)
print(len(halluc), 'trajectories |', sorted(set(read_pdb(halluc[0]).chain)),
      'chains in the hallucinated complex')

In [ ]:
%%group 2
# the same judge as arm A. The designability test reads the chains by position, so the target must be
# chain A and the binder chain B (chains_to_AB puts the longest chain first and renames both if needed).
for f in halluc:
    chains_to_AB(f)
print('chains:', chain_sizes(halluc[0]), '| contigs:', binder_contigs(halluc[0]))

hall = run_designability(halluc, f'hallucination_{MEMBER}', contigs=binder_contigs(halluc[0]),
                         num_seqs=NUM_SEQS, initial_guess=True, multimer=True)
hall['method'] = 'hallucination'
display(hall[['backbone', 'n', 'plddt', 'i_ptm', 'i_pae', 'rmsd']].round(2))

## Collect everything

We add to the metrics what we can measure on the structures themselves: how much of the target each binder
touches, how many of the hotspots it reaches, and how helical it is (the fraction of residues whose
C-alpha(i)-C-alpha(i+4) distance is that of an alpha helix).

In [ ]:
%%group 2
def helix_fraction(pdb, chain):
    a = read_pdb(pdb)
    ca = a[(a.name == 'CA') & (a.chain == chain)][['x', 'y', 'z']].values
    if len(ca) < 5:
        return np.nan
    d = np.linalg.norm(ca[4:] - ca[:-4], axis=1)
    return float((d < 6.5).mean())

both = pd.concat([diff, hall], ignore_index=True)
chains = [guess_chains(p) for p in both.pdb]        # (target, binder) in each predicted complex
both['epitope'] = [contacts(p, t, b) for p, (t, b) in zip(both.pdb, chains)]
both['n_contacts'] = [len(e) for e in both.epitope]
both['hotspots_hit'] = [len(set(e) & {56, 58, 113, 122, 123}) for e in both.epitope]
both['helix'] = [helix_fraction(p, b) for p, (t, b) in zip(both.pdb, chains)]
both['binder_seq'] = [sequence_of(read_pdb(p), chain=b) for p, (t, b) in zip(both.pdb, chains)]
both['member'], both['binder_len'], both['hotspots'] = MEMBER, BINDER_LEN, HOTSPOTS
save_json(both.drop(columns=['pdb']).to_dict(orient='list'),
          os.path.join(local_path, 'results', f'binders_{MEMBER}_{BINDER_LEN}.json'))
display(both.groupby('method')[['plddt', 'i_ptm', 'i_pae', 'rmsd', 'n_contacts', 'helix']].mean().round(2))

In [ ]:
%%group 2
install_boltz()      # a few minutes the first time

In [ ]:
%%group 2
# an independent opinion on the best design of each arm (3-5 minutes each)
rows = []
for method, d in both.groupby('method'):
    best = d.sort_values('i_pae').iloc[0]
    y = boltz_yaml(os.path.join(run_dir, f'boltz_{method}.yaml'),
                   chains={'B': best.binder_seq, 'T': pdl1_seq})
    c = run_boltz(y, out_dir=os.path.join(run_dir, 'boltz'))
    rows.append(dict(method=method, member=MEMBER, seq=best.binder_seq, af2_i_pae=float(best.i_pae),
                     af2_i_ptm=float(best.i_ptm), boltz_iptm=c['iptm'],
                     boltz_plddt=100 * c['complex_plddt'], pdb=c['pdb']))
    print(method, 'AF2 i_pae', round(best.i_pae, 1), '-> Boltz-2 iptm', round(c['iptm'], 2))
boltz = pd.DataFrame(rows)
save_json(boltz.drop(columns=['pdb']).to_dict(orient='list'),
          os.path.join(local_path, 'results', f'boltz_{MEMBER}_{BINDER_LEN}.json'))

# GROUP 2, Part 2: the report

Run `sync_from_drive()` first: the cells below use the results of all the members.

In [ ]:
%%group 2
sync_from_drive()
allb = load_results(os.path.join(local_path, 'results', 'binders_*.json'), 'designs')
print(len(allb), 'designs;', allb.member.nunique(), 'members; lengths', sorted(allb.binder_len.unique()))
display(allb.groupby(['method', 'binder_len']).size().rename('designs').reset_index())

## 1. The two pipelines

**Figure 1**: a scheme of the two routes you followed, with the best design of each next to PD-L1.

In [ ]:
%%group 2
# the funnel of each method, side by side
for m, d in allb.groupby('method'):
    f, thr = apply_filters(d, 'binder')
    funnel(f, thr, title=m, name=f'funnel_{m}')

**Question 1.** Which of the two methods produced more designs that pass the filters, and at what cost in
computer time? Compute the time per *accepted* design, not per design: it is the number that matters when you
plan a campaign.

*(your answer)*

## 2. Are they the same proteins?

**Figure 2**: the metrics of the two methods compared (for example `i_pae` against `plddt`, with one colour
per method), plus the distribution of the helical fraction.

In [ ]:
%%group 2
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for m, d in allb.groupby('method'):
    axes[0].scatter(d.plddt, d.i_pae, label=m, alpha=0.7)
    axes[1].hist(d.helix.dropna(), bins=np.linspace(0, 1, 11), alpha=0.6, label=m)
    axes[2].hist(d.n_contacts, bins=12, alpha=0.6, label=m)
axes[0].axhline(10, ls='--', c='k'); axes[0].set_xlabel('pLDDT'); axes[0].set_ylabel('interface pAE')
axes[1].set_xlabel('helical fraction'); axes[2].set_xlabel('residues of PD-L1 touched')
for a in axes: a.legend(fontsize=8)
plt.tight_layout(); save_figure(fig, 'diffusion_vs_hallucination')

**Question 2.** Do the two methods make the same kind of protein? Look at the fold (helices?), at the size of
the interface and at the sequences themselves: hallucinated sequences are sometimes full of a few favourite
amino acids. Count the composition if you suspect it.

*(your answer)*

## 3. Where do they bind, and how does that compare with a real one?

[8ZNL](https://www.rcsb.org/structure/8ZNL) is a de novo binder of PD-L1 designed by others, 58 residues,
with an experimental structure and picomolar affinity. The cell computes which residues of PD-L1 it uses and
compares with your designs.

In [ ]:
%%group 2
ref = fetch_pdb('8znl', out=os.path.join(local_path, '8znl.pdb'))
print('chains in 8ZNL:', chain_sizes(ref), '(the binder is the short one, 58 residues)')

In [ ]:
%%group 2
TARGET_CH, BINDER_CH = guess_chains(ref)     # the long chain is PD-L1, the short one the binder
epi_ref = set(contacts(ref, TARGET_CH, BINDER_CH))
print(f'in 8ZNL, PD-L1 is chain {TARGET_CH} and the designed binder chain {BINDER_CH}')
print('residues of PD-L1 it uses:', sorted(epi_ref) or 'none found - check the chains above')

from collections import Counter
count = Counter(r for e in allb.epitope for r in e)
res = sorted(count)
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(res)), [count[r] for r in res],
       color=['#ee854a' if r in epi_ref else '0.7' for r in res])
ax.set_xticks(range(len(res))); ax.set_xticklabels(res, rotation=90, fontsize=6)
ax.set_xlabel('residue of PD-L1'); ax.set_ylabel('designs in contact')
ax.set_title('orange = also used by the published de novo binder (8ZNL)')
save_figure(fig, 'epitope_vs_8znl')

**Figure 3**: the histogram above, and a 3D view of your best design superposed on 8ZNL.

**Question 3.** Did the hotspots do their job? Is there a difference between the two methods in where they
land? And how does the fraction of the PD-1 surface that you cover compare with the published binder?

*(your answer)*

## 4. The second opinion

**Figure 4**: AlphaFold2 against Boltz-2 for the designs you tested with both.

In [ ]:
%%group 2
allboltz = load_results(os.path.join(local_path, 'results', 'boltz_*.json'), 'Boltz-2 predictions')
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for m, d in allboltz.groupby('method'):
    ax.scatter(d.af2_i_pae, d.boltz_iptm, label=m)
ax.axvline(10, ls='--', c='k'); ax.axhline(0.5, ls='--', c='k')
ax.set_xlabel('AlphaFold2 interface pAE'); ax.set_ylabel('Boltz-2 iptm'); ax.legend(fontsize=8)
save_figure(fig, 'af2_vs_boltz')
display(allboltz.drop(columns=['pdb'], errors='ignore').round(3))

**Question 4.** Does the second model confirm the first? Is one of the two design methods more often
contradicted by Boltz-2 than the other, and why would that be expected?

*(your answer)*

## 5. Novelty and the decision

Run your best design (chain B alone) through [search.foldseek.com](https://search.foldseek.com).

**Question 5.** Is the fold new? Then: you can order three designs. Which do you choose, and which numbers did
you use? Name one property that decides whether a mini-binder is usable in practice and that none of these
numbers measures.

*(your answer)*

## 6. Going further (optional)

* **BindCraft** ([Colab](https://colab.research.google.com/github/martinpacesa/BindCraft/blob/main/notebooks/BindCraft.ipynb))
  is the production version of arm B: same idea, much better filters and sampling. Run 2-3 trajectories on
  the same target and compare what it produces with your own hallucinated designs. Do not expect accepted
  designs on a free GPU: the point is to see the protocol and its filter suite.
* Take your best design and generate variations with partial diffusion (`diffuser.partial_T=10`), then check
  whether the metrics improve.
* Design a binder against a *different* patch of PD-L1 (one that does not overlap the PD-1 surface): does it
  work equally well? Not all surfaces of a protein are designable.

# GROUP 3, Part 1: building a protein around a chemical site

A binder only has to stick to something. A **functional site** has to hold a few atoms in a precise geometry:
the three histidines that grip the zinc of carbonic anhydrase sit at distances and angles that are worth
millions of years of evolution, and moving one of them by half an Ångström switches the chemistry off.

The exercise: take such a site out of its natural protein and **build a completely new protein around it**
(*motif scaffolding*). This is how de novo enzymes are made today, and it asks a different question from the
binder exercises: not "does it stick?" but "did the geometry survive?".

Three steps, and the second and third use tools we have not used yet:

1. **RFdiffusion** generates a new protein that contains the motif residues in their original geometry. It
   knows nothing about the metal or the substrate: it only sees the backbone of the residues you kept. (This
   limitation is exactly why **RFdiffusion2** was built in 2025: it reasons at the level of the catalytic
   atoms, without even being told in which order the residues appear in the sequence.)
2. **LigandMPNN** writes the sequence. Unlike ProteinMPNN it *sees* the ligand, the metal and the water around
   the protein, so the residues it chooses next to the site are chemically sensible. You will design the same
   backbone with both and compare.
3. **Boltz-2** predicts the structure of the designed sequence **together with the ligand**, from sequence and
   chemistry alone. If the ligand comes back in the pocket you built, the design holds up; if it floats
   somewhere else, you built a protein, not a site.

In [ ]:
%%group 3
# LigandMPNN. Three practical details, all of them typical:
#  * it needs dm-tree and prody, which have no version for the Python of Colab, so it runs in the
#    environment of RFdiffusion (Python 3.12) that the setup cells already built;
#  * one of the files it carries inside uses `np.int`, an alias numpy removed in 2022: one line to patch;
#  * its script downloads two dozen sets of weights, and we only use two.
LMPNN = os.path.join(WORK, 'LigandMPNN')
if not os.path.isfile(os.path.join(LMPNN, 'run.py')):          # also repairs an interrupted clone
    shutil.rmtree(LMPNN, ignore_errors=True)
    sh(f'cd {WORK} && git clone -q https://github.com/dauparas/LigandMPNN.git', tries=2)
    if not os.path.isfile(os.path.join(LMPNN, 'run.py')):
        raise RuntimeError('could not clone LigandMPNN: check the network and run the cell again')
constants = os.path.join(LMPNN, 'openfold', 'np', 'residue_constants.py')
if 'dtype=np.int)' in open(constants).read():                  # the alias numpy removed years ago
    sh(f"sed -i 's/dtype=np\\.int)/dtype=int)/g' {constants}")
if not imports(['prody', 'tree'], RFD_PYTHON):
    sh(f'{UV} pip install -q --python "{RFD_PYTHON}" prody ml-collections dm-tree biopython', tries=2)

os.makedirs(os.path.join(LMPNN, 'model_params'), exist_ok=True)
for w in ['ligandmpnn_v_32_010_25.pt', 'proteinmpnn_v_48_020.pt']:
    if not os.path.isfile(os.path.join(LMPNN, 'model_params', w)):
        sh(f'wget -q https://files.ipd.uw.edu/pub/ligandmpnn/{w} -O {LMPNN}/model_params/{w}', tries=2)

print('weights    :', len(glob.glob(os.path.join(LMPNN, 'model_params', '*.pt'))), 'files')
print('LigandMPNN :', 'ready' if imports(['prody', 'tree'], RFD_PYTHON) else 'the libraries are missing')

## Choose your site

Two sites, both small, both catalytic. Take one each (or in pairs) and make sure the group covers both: the
comparison between a **metal** site and an **organic** one is part of the report.

**A. The zinc of carbonic anhydrase II** ([1CA2](https://www.rcsb.org/structure/1CA2)). One of the fastest
enzymes known: it turns CO2 into bicarbonate. The zinc is held by three histidines, His94, His96 and His119,
and activates a water molecule into a hydroxide. The motif is three residues, two of them adjacent.

**B. The Tyr/Asp dyad of ketosteroid isomerase** ([1C7H](https://www.rcsb.org/structure/1C7H)). It moves a
double bond along a steroid; a tyrosine and an aspartate do the acid-base chemistry inside a very hydrophobic
pocket. The motif is two residues far apart in the sequence.

The cell below downloads the structure and prints what is inside: the ligands (HETATM) and the residues around
them. **Look at the output and fill in the next cell yourself** - finding the catalytic residues in a
structure is part of the exercise, and the numbering in the PDB file is not always the one in the papers.

In [ ]:
%%group 3
PDB_ID = '1ca2'                 # '1ca2' (zinc) or '1c7h' (steroid isomerase)
raw = fetch_pdb(PDB_ID, chains='A', keep_hetero=('ZN', 'HOH', 'EQU', 'AND', 'DEX', 'EST'),
                out=os.path.join(local_path, PDB_ID + '_full.pdb'))
atoms = read_pdb(raw)
het = atoms[atoms.hetero & (atoms.resname != 'HOH')]
print('ligands:', het.groupby(['resname', 'resi']).size().to_dict())
for (rn, ri), g in het.groupby(['resname', 'resi']):
    d = np.linalg.norm(atoms[['x','y','z']].values[:, None] - g[['x','y','z']].values[None], axis=-1).min(1)
    near = atoms[(d < 4.5) & (~atoms.hetero)][['resname', 'resi']].drop_duplicates()
    print(f'\nresidues within 4.5 A of {rn}{ri}:')
    print('  ' + ', '.join(f'{r.resname}{r.resi}' for _, r in near.iterrows()))

In [ ]:
%%group 3
MEMBER   = 'name'               # your name: it labels your files in the group folder
LIGAND   = 'ZN'                 # the ligand code printed above ('ZN' for 1ca2)
MOTIF    = [(94, 96), (119, 119)]   # the catalytic residues, as (first, last) blocks in the pdb numbering
CONTIGS  = '15-30/A94-96/8-25/A119-119/15-30'   # the new protein around them: invent, keep, invent, keep, invent
NUM_DESIGNS = 8

assert all(f'A{a}-{b}' in CONTIGS for a, b in MOTIF), \
    'CONTIGS and MOTIF do not describe the same residues: correct one of the two'
motif_resi = [r for a, b in MOTIF for r in range(a, b + 1)]
run_dir = os.path.join(local_path, f'{PDB_ID}_{MEMBER}')
os.makedirs(run_dir, exist_ok=True)

# the input for RFdiffusion: the protein only (it cannot see the ligand); the ligand is kept aside for later
site = fetch_pdb(PDB_ID, chains='A', out=os.path.join(run_dir, 'site_protein.pdb'))
print(f'{len(atoms[atoms.resname == LIGAND])} atoms of {LIGAND} will be put back into the designs later')
show(raw, sticks=[('A', r) for r in motif_resi])

## Generate the scaffolds

The contigs are the heart of this exercise. `15-30/A94-96/8-25/A119-119/15-30` means: invent 15 to 30
residues, then keep residues 94-96 of chain A exactly as they are, then invent 8 to 25, then keep residue 119,
then invent 15 to 30 more. The ranges let RFdiffusion choose the lengths that make a protein possible around
your geometry; if nothing works, widen them.

About 2-4 minutes per design.

In [ ]:
%%group 3
scaffolds = run_rfdiffusion(f'scaffold_{MEMBER}', contigs=CONTIGS, pdb=site, num_designs=NUM_DESIGNS, steps=50)
show(scaffolds[0], colors=('salmon',))

RFdiffusion renumbers its output from 1, so we need to know where the motif ended up in each design. It writes
this in the `.trb` file that accompanies every structure: the correspondence between the residues of the input
(`con_ref_pdb_idx`) and those of the design (`con_hal_pdb_idx`). We use it for everything that follows: to
measure whether the geometry survived, to put the ligand back, and to tell LigandMPNN which residues it must
not change.

In [ ]:
%%group 3
def motif_map(pdb):
    """for a design, the correspondence motif residue in the original -> residue number in the design"""
    trb = pickle.load(open(pdb.replace('.pdb', '.trb'), 'rb'))
    ref = [r for _, r in trb['con_ref_pdb_idx']]
    hal = [r for _, r in trb['con_hal_pdb_idx']]
    return dict(zip(ref, hal))

for f in scaffolds[:2]:
    m = motif_map(f)
    print(os.path.basename(f), '->', m, ' rmsd of the motif:',
          round(ca_rmsd(f, site, resi1=list(m.values()), resi2=list(m.keys())), 2), 'A')

## Sequences: ProteinMPNN against LigandMPNN

First the usual designability test (ProteinMPNN + AlphaFold2), which gives the metrics of a monomer and tells
us whether the scaffold is foldable at all. Then LigandMPNN on the same backbones, this time **with the ligand
present** and with the motif residues frozen.

In [ ]:
%%group 3
# the contigs of the designability test are not the ones we asked RFdiffusion for: the lengths between the
# motif residues were chosen at random inside the ranges, so we read them back from the .trb of each design
# (motif residues are kept by ProteinMPNN, the rest is redesigned)
print('contigs of the first design:', motif_contigs(scaffolds[0]))
res = pd.concat([run_designability([s], f'scaffold_{MEMBER}', contigs=motif_contigs(s), num_seqs=4)
                 for s in scaffolds], ignore_index=True)
res['method'] = 'ProteinMPNN'
display(res[['backbone', 'n', 'mpnn', 'plddt', 'ptm', 'pae', 'rmsd']].round(2))

In [ ]:
%%group 3
# put the ligand back: superpose the original site (ligand included) on each design, using the motif residues
def transplant(design):
    m = motif_map(design)
    moved = design.replace('.pdb', '_site.pdb')
    rms = superpose_onto(raw, design, mobile_resi=list(m.keys()), ref_resi=list(m.values()), out_pdb=moved)
    # the design plus the ligand alone, which is what LigandMPNN and Boltz need
    lig = read_pdb(moved)
    lig = lig[lig.resname == LIGAND]
    out = design.replace('.pdb', '_lig.pdb')
    d = read_pdb(design)
    write_pdb(pd.concat([d, lig]), out)
    return out, rms, m

complexes = {}
for f in scaffolds:
    out, rms, m = transplant(f)
    complexes[f] = dict(pdb=out, motif_rmsd=rms, map=m)
show(list(complexes.values())[0]['pdb'],
     sticks=[('A', r) for r in list(complexes.values())[0]['map'].values()])

In [ ]:
%%group 3
def run_ligandmpnn(pdb, out, fixed, n=8, model='ligand_mpnn', seed=1, rerun=False):
    """design the sequence with the ligand in the structure, keeping the motif residues fixed.
    Returns one row per sequence, with the confidence that LigandMPNN reports for it."""
    name = os.path.basename(pdb).replace('.pdb', '')
    fasta = os.path.join(out, 'seqs', name + '.fa')
    fixed_str = ' '.join(f'A{r}' for r in fixed)        # LigandMPNN names residues as <chain><number>
    run(f'cd {LMPNN} && {RFD_PYTHON} run.py --model_type "{model}" --pdb_path {pdb} --out_folder {out} '
        f'--fixed_residues "{fixed_str}" --batch_size {n} --number_of_batches 1 --seed {seed}',
        log=os.path.join(out, 'ligandmpnn.log'), rerun=rerun, done=[fasta])
    lines = [l.strip() for l in open(fasta) if l.strip()]
    rows = []
    for header, seq in zip(lines[::2], lines[1::2]):
        if 'id=' not in header:                          # the first entry is the sequence of the input
            continue
        values = dict(re.findall(r'(\w+)=([-\d.]+)', header))
        rows.append(dict(seq=seq.replace(':', ''), model=model,
                         **{k: float(v) for k, v in values.items()
                            if k in ('overall_confidence', 'ligand_confidence', 'seq_rec')}))
    return pd.DataFrame(rows)

lmpnn = {}
for i, (f, c) in enumerate(complexes.items()):
    seqs = run_ligandmpnn(c['pdb'], os.path.join(run_dir, 'lmpnn_' + os.path.basename(f)[:-4]),
                          fixed=list(c['map'].values()), seed=i + 1)
    lmpnn[f] = seqs
    print(os.path.basename(f), seqs.seq.iloc[0], f'(confidence {seqs.overall_confidence.iloc[0]:.2f})')

In [ ]:
%%group 3
# which residues does each method put around the ligand?
def around_ligand(pdb, cutoff=6.0):
    a = read_pdb(pdb)
    lig = a[a.resname == LIGAND]
    prot = a[~a.hetero]
    d = np.linalg.norm(prot[['x','y','z']].values[:, None] - lig[['x','y','z']].values[None], axis=-1).min(1)
    return sorted(set(prot.resi.values[d < cutoff]))

rows = []
for f, c in complexes.items():
    pocket = around_ligand(c['pdb'])
    pm = res[res.backbone == os.path.basename(f)[:-4]].sort_values('rmsd').iloc[0].seq
    lm = lmpnn[f].seq.iloc[0]
    rows.append(dict(design=os.path.basename(f)[:-4], lig_pdb=c['pdb'],
                     motif_rmsd=c['motif_rmsd'], pocket=pocket,
                     proteinmpnn=''.join(pm[r-1] for r in pocket if r <= len(pm)),
                     ligandmpnn=''.join(lm[r-1] for r in pocket if r <= len(lm)),
                     seq_proteinmpnn=pm, seq_ligandmpnn=lm, member=MEMBER, site=PDB_ID, ligand=LIGAND))
pockets = pd.DataFrame(rows)
save_json(pockets.to_dict(orient='list'), os.path.join(local_path, 'results', f'pockets_{PDB_ID}_{MEMBER}.json'))
save_json(res.drop(columns=['pdb']).assign(member=MEMBER, site=PDB_ID).to_dict(orient='list'),
          os.path.join(local_path, 'results', f'designs_{PDB_ID}_{MEMBER}.json'))
display(pockets[['design', 'motif_rmsd', 'pocket', 'proteinmpnn', 'ligandmpnn']])

## The real test: does the ligand come back?

Everything so far was done *knowing* where the ligand is. Boltz-2 is given only the designed sequence and the
chemical identity of the ligand, and has to put the two together. We then measure how far its ligand is from
the position you designed, after superposing the two proteins.

About 3-6 minutes per sequence: choose your best two or three designs.

In [ ]:
%%group 3
install_boltz()      # a few minutes the first time

In [ ]:
%%group 3
rows = []
for _, p in pockets.sort_values('motif_rmsd').head(2).iterrows():
    for method, seq in [('ProteinMPNN', p.seq_proteinmpnn), ('LigandMPNN', p.seq_ligandmpnn)]:
        seq = seq.replace('/', '')          # ProteinMPNN separates chains with '/': this one is a monomer
        y = boltz_yaml(os.path.join(run_dir, f'{p.design}_{method}.yaml'),
                       chains={'A': seq}, ligand=('L', LIGAND))
        c = run_boltz(y, out_dir=os.path.join(run_dir, 'boltz'))
        # distance between the ligand predicted by Boltz and the one you designed, after superposing the proteins
        design_pdb = p.lig_pdb            # the design with the ligand put back in place
        moved = superpose_onto(c['pdb'], design_pdb, mobile_chain='A', ref_chain='A',
                               mobile_resi=range(1, len(seq) + 1), ref_resi=range(1, len(seq) + 1),
                               out_pdb=c['pdb'].replace('.pdb', '_fit.pdb'))
        a, b = read_pdb(c['pdb'].replace('.pdb', '_fit.pdb')), read_pdb(design_pdb)
        la = a[a.resname.isin([LIGAND])][['x','y','z']].values
        lb = b[b.resname.isin([LIGAND])][['x','y','z']].values
        dist = float(np.linalg.norm(la.mean(0) - lb.mean(0))) if len(la) and len(lb) else np.nan
        rows.append(dict(design=p.design, method=method, member=MEMBER, site=PDB_ID,
                         plddt=100 * c['complex_plddt'], ligand_iptm=c.get('ligand_iptm', np.nan),
                         ligand_shift=dist, motif_rmsd=p.motif_rmsd))
        print(p.design, method, 'ligand displaced by', round(dist, 1), 'A')
cofold = pd.DataFrame(rows)
save_json(cofold.to_dict(orient='list'), os.path.join(local_path, 'results', f'cofold_{PDB_ID}_{MEMBER}.json'))
display(cofold.round(2))

# GROUP 3, Part 2: the report

Run `sync_from_drive()` first: the cells below use the results of all the members, on both sites.

In [ ]:
%%group 3
sync_from_drive()
D = lambda pat, what: load_results(os.path.join(local_path, 'results', pat), what)
alld = D('designs_*.json', 'designs')
allp = D('pockets_*.json', 'pockets')
allc = D('cofold_*.json', 'co-folding results')
print(len(alld), 'designs;', 'sites:', sorted(allp.site.unique()))

## 1. Did the geometry survive?

**Figure 1**: the procedure and your best design, with the motif residues and the ligand shown as sticks,
next to the original enzyme.

In [ ]:
%%group 3
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for s, d in allp.groupby('site'):
    axes[0].hist(d.motif_rmsd, bins=12, alpha=0.6, label=s)
axes[0].axvline(1.0, ls='--', c='k'); axes[0].set_xlabel('rmsd of the motif (Å)'); axes[0].legend(fontsize=8)
for s, d in alld.groupby('site'):
    axes[1].scatter(d.plddt, d.rmsd, label=s, alpha=0.7)
axes[1].axhline(2, ls='--', c='k'); axes[1].set_xlabel('pLDDT'); axes[1].set_ylabel('rmsd design/prediction (Å)')
axes[1].legend(fontsize=8); plt.tight_layout(); save_figure(fig, 'motif_and_folding')

In [ ]:
%%group 3
for s, d in alld.groupby('site'):
    f, thr = apply_filters(d, 'monomer')
    funnel(f, thr, title=s, name=f'funnel_{s}')

**Question 1.** Two different things can fail here: the protein may not fold (pLDDT, rmsd design/prediction),
or it may fold while losing the geometry of the site (rmsd of the motif). Which happened more often, and was
it the same for the two sites? The zinc motif is three residues in two blocks, the dyad is two residues far
apart: does that explain the difference?

*(your answer)*

## 2. What LigandMPNN knows that ProteinMPNN does not

**Figure 2**: the residues that the two methods put around the ligand (the table of the pockets), as a figure:
for example the amino acid composition of the pocket for the two methods, or a 3D view with the two sequences
mapped on the same backbone.

In [ ]:
%%group 3
from collections import Counter
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, col in zip(axes, ['proteinmpnn', 'ligandmpnn']):
    c = Counter(''.join(allp[col].astype(str)))
    aa = sorted(c)
    ax.bar(aa, [c[a] for a in aa], color='#4878d0' if col == 'ligandmpnn' else '0.7')
    ax.set_title(col); ax.set_xlabel('residues within 6 Å of the ligand')
plt.tight_layout(); save_figure(fig, 'pocket_composition')
display(allp[['site', 'design', 'motif_rmsd', 'proteinmpnn', 'ligandmpnn']].head(20))

**Question 2.** Do the two methods choose different residues next to the ligand? Say precisely what changes
(charges next to the metal? hydrophobic residues around the steroid?) and whether it makes chemical sense. Be
careful: a difference of one or two residues over a handful of designs is not a result; look at all the designs
of the group.

*(your answer)*

## 3. The test that counts

**Figure 3**: for the designs you co-folded, how far the ligand ended up from where you put it, for sequences
written by ProteinMPNN and by LigandMPNN.

In [ ]:
%%group 3
fig, ax = plt.subplots(figsize=(5, 3.5))
for i, (m, d) in enumerate(allc.groupby('method')):
    ax.scatter(np.full(len(d), i) + np.random.uniform(-.1, .1, len(d)), d.ligand_shift, label=m)
ax.set_xticks(range(allc.method.nunique())); ax.set_xticklabels(sorted(allc.method.unique()))
ax.axhline(2, ls='--', c='k'); ax.set_ylabel('displacement of the ligand (Å)')
save_figure(fig, 'ligand_placement')
display(allc.round(2))

**Question 3.** When Boltz-2 is given only the sequence and the chemical formula of the ligand, does it put the
ligand in the site you designed? Does the sequence written with the ligand in view do better? And what does a
displacement of 10 Å mean: that the design is wrong, or that the predictor cannot do this?

*(your answer)*

## 4. Is it a new protein?

Take your best design to [search.foldseek.com](https://search.foldseek.com).

**Question 4.** Is the scaffold new, or did RFdiffusion rebuild something close to the original enzyme (or to
another protein with the same fold)? How much sequence identity is there with the closest hit?

*(your answer)*

## 5. What would be missing to have an enzyme

**Question 5.** Suppose your best design folds, holds its metal and places the ligand correctly. List what
still separates it from a working enzyme, and say which of those things could be checked *in silico*, with the
tools you have met in this course (docking in T03, molecular dynamics in T04-T06, quantum chemistry in T07),
and which require the laboratory.

*(your answer)*

## 6. Going further (optional)

* Run the same motif with **wider contigs** (for example `30-60/A94-96/10-40/A119-119/30-60`): do bigger
  proteins hold the geometry better?
* Add a second shell: keep also the residues that hold the catalytic ones in place (in carbonic anhydrase,
  Gln92, Glu117, Thr199) and see whether the motif rmsd improves.
* Read the abstract of **RFdiffusion2** (Nat. Methods 2025) and say, in three lines, which of the problems you
  met it is designed to solve.

# Before you hand in

* Every design you discuss must be identifiable (which member, which parameters, which run) and must come
  with its numbers.
* Report the **success rate** of your group, not only the best design.
* Say at least once where the method you used is known to be wrong, and what you would do to find out.